<a href="https://colab.research.google.com/github/lekhanh5102005-commits/bft65b-fdic-bank-failure/blob/main/Big_Data_Analysis_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BFT65B Final Report - FDIC bank failure early-warning pipeline
End-to-end pipeline: download FDIC BankFind data (1984Q1-2026Q2), Parquet storage, data-quality report, Polars/Spark processing, point-in-time features and labels, walk-forward validation, and one frozen test evaluation.

**How to run:** Google Colab + Google Drive. Set `WORK_DIR` in the second code cell. Cells run top to bottom.

**Test set:** the evaluation cell under "Final test evaluation" creates `TEST_USED.lock` and refuses to run if it already exists. Do not "Run all" on a copy of the Drive folder that already contains the lock; delete the lock only to reproduce everything from scratch on a clean copy.

## Part 1 - Data acquisition and storage

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

WORK_DIR = Path("/content/drive/MyDrive/bft65b_fdic_project")
DATA_DIR = WORK_DIR / "data_raw" / "fdic"
BENCH_DIR = WORK_DIR / "benchmarks"

WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
BENCH_DIR.mkdir(parents=True, exist_ok=True)

print("Work dir:", WORK_DIR)

Work dir: /content/drive/MyDrive/bft65b_fdic_project


In [ ]:
import glob, shutil

matches = glob.glob("/content/fdic_download_starter*.py")
print("Tìm thấy:", matches)

if matches:
    shutil.move(matches[0], "/content/fdic_download_starter.py")
    print("Đã đổi tên thành /content/fdic_download_starter.py")
else:
    print("Chưa có file trong /content — hãy bấm biểu tượng thư mục bên trái để upload file nhé!")

Tìm thấy: []
Chưa có file trong /content — hãy bấm biểu tượng thư mục bên trái để upload file nhé!


In [ ]:
!python /content/drive/MyDrive/bft65b_fdic_project/fdic_download_starter.py --start 1984-03-31 --end 2026-06-30 --out /content/drive/MyDrive/bft65b_fdic_project/data_raw/fdic


[1/170] 19840331: already complete, skipped
[2/170] 19840630: already complete, skipped
[3/170] 19840930: already complete, skipped
[4/170] 19841231: already complete, skipped
[5/170] 19850331: already complete, skipped
[6/170] 19850630: already complete, skipped
[7/170] 19850930: already complete, skipped
[8/170] 19851231: already complete, skipped
[9/170] 19860331: already complete, skipped
[10/170] 19860630: already complete, skipped
[11/170] 19860930: already complete, skipped
[12/170] 19861231: already complete, skipped
[13/170] 19870331: already complete, skipped
[14/170] 19870630: already complete, skipped
[15/170] 19870930: already complete, skipped
[16/170] 19871231: already complete, skipped
[17/170] 19880331: already complete, skipped
[18/170] 19880630: already complete, skipped
[19/170] 19880930: already complete, skipped
[20/170] 19881231: already complete, skipped
[21/170] 19890331: already complete, skipped
[22/170] 19890630: already complete, skipped
[23/170] 19890930:

In [ ]:
import time
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

WORK_DIR = Path("/content/drive/MyDrive/bft65b_fdic_project")
DATA_DIR = WORK_DIR / "data_raw" / "fdic"
BENCH_DIR = WORK_DIR / "benchmarks"
BENCH_DIR.mkdir(parents=True, exist_ok=True)

files = sorted((DATA_DIR / "financials").glob("*.parquet"))

# 1. Kiểm tra cột nào bị lệch kiểu dữ liệu (int64 vs float64 vs null) giữa các quý
col_types = {}
for f in files:
    schema = pq.read_schema(f)
    for field in schema:
        col_types.setdefault(field.name, set()).add(str(field.type))

drifted_cols = {c: sorted(t) for c, t in col_types.items() if len(t) > 1}
print("=== CÁC CỘT BỊ SCHEMA DRIFT GIỮA CÁC QUÝ ===")
for c, t in drifted_cols.items():
    print(f"  {c:<10}: {t}")

# 2. Định nghĩa unified schema (cột định danh giữ string/int64, toàn bộ chỉ tiêu tài chính ép float64)
str_cols = {"REPDTE", "STNAME"}
int_cols = {"CERT"}

def read_unified_parquet(parquet_files, columns=None):
    tables = []
    for f in parquet_files:
        tbl = pq.read_table(f, columns=columns)
        new_fields = []
        for field in tbl.schema:
            if field.name in str_cols:
                new_fields.append(pa.field(field.name, pa.string()))
            elif field.name in int_cols:
                new_fields.append(pa.field(field.name, pa.int64()))
            else:
                new_fields.append(pa.field(field.name, pa.float64()))
        tables.append(tbl.cast(pa.schema(new_fields)))
    return pa.concat_tables(tables).to_pandas()

# 3. Đo đọc Parquet hiện tại (170 file quý, nén snappy)
t0 = time.perf_counter()
df_fin = read_unified_parquet(files)
t_read_pq_snappy = time.perf_counter() - t0

# 4. Lưu thêm 1 bản Parquet hợp nhất nén ZSTD (đã chuẩn hóa schema để đọc cực nhanh)
pq_zstd_path = BENCH_DIR / "financials_unified_zstd.parquet"
t0 = time.perf_counter()
df_fin.to_parquet(pq_zstd_path, index=False, compression="zstd")
t_write_pq_zstd = time.perf_counter() - t0

t0 = time.perf_counter()
_ = pd.read_parquet(pq_zstd_path)
t_read_pq_zstd = time.perf_counter() - t0

# 5. Lưu sang CSV thô để benchmark đối chứng
csv_path = BENCH_DIR / "financials_raw.csv"
t0 = time.perf_counter()
df_fin.to_csv(csv_path, index=False)
t_write_csv = time.perf_counter() - t0

t0 = time.perf_counter()
_ = pd.read_csv(csv_path, low_memory=False)
t_read_csv_full = time.perf_counter() - t0

# 6. Đo đọc chọn lọc 6 cột (column pruning)
cols_sub = ["CERT", "REPDTE", "ASSET", "EQ", "ROA", "NCLNLS"]
t0 = time.perf_counter()
_ = read_unified_parquet(files, columns=cols_sub)
t_read_pq_snappy_sub = time.perf_counter() - t0

t0 = time.perf_counter()
_ = pd.read_parquet(pq_zstd_path, columns=cols_sub)
t_read_pq_zstd_sub = time.perf_counter() - t0

t0 = time.perf_counter()
_ = pd.read_csv(csv_path, usecols=cols_sub, low_memory=False)
t_read_csv_sub = time.perf_counter() - t0

# 7. Tính dung lượng (MB)
snappy_mb = sum(f.stat().st_size for f in files) / (1024**2)
zstd_mb = pq_zstd_path.stat().st_size / (1024**2)
csv_mb = csv_path.stat().st_size / (1024**2)

summary_part1 = pd.DataFrame([
    {"Format": "Raw CSV", "Size_MB": round(csv_mb, 2), "Read_Full_s": round(t_read_csv_full, 2), "Read_6Cols_s": round(t_read_csv_sub, 2)},
    {"Format": "Parquet (170 quarter files, Snappy)", "Size_MB": round(snappy_mb, 2), "Read_Full_s": round(t_read_pq_snappy, 2), "Read_6Cols_s": round(t_read_pq_snappy_sub, 2)},
    {"Format": "Parquet (Unified schema, Zstd)", "Size_MB": round(zstd_mb, 2), "Read_Full_s": round(t_read_pq_zstd, 2), "Read_6Cols_s": round(t_read_pq_zstd_sub, 2)},
])
print("\n=== BENCHMARK STORAGE (PART 1) ===")
print(summary_part1.to_string(index=False))

# Xóa file CSV tạm để tiết kiệm dung lượng Google Drive (giữ lại file unified parquet nếu cần dùng tiếp)
csv_path.unlink(missing_ok=True)

=== CÁC CỘT BỊ SCHEMA DRIFT GIỮA CÁC QUÝ ===
  LNCI      : ['double', 'int64']
  SC        : ['double', 'int64']
  NCLNLS    : ['double', 'int64']
  LNRE      : ['double', 'int64']
  LNLSNET   : ['double', 'int64']
  DEPDOM    : ['double', 'int64']
  LNLSGR    : ['double', 'int64']
  DEP       : ['double', 'int64']
  NAASSET   : ['double', 'int64']
  LNCON     : ['double', 'int64']
  RBCRWAJ   : ['double', 'int64']
  P9ASSET   : ['double', 'int64']
  CHBAL     : ['double', 'int64']
  P3ASSET   : ['int64', 'null']

=== BENCHMARK STORAGE (PART 1) ===
                             Format  Size_MB  Read_Full_s  Read_6Cols_s
                            Raw CSV   422.52        16.80          7.57
Parquet (170 quarter files, Snappy)   236.76         5.36          3.60
     Parquet (Unified schema, Zstd)   144.31         4.23          0.39


### Part 1 - Final storage layout (partitioned by year) and partition-pruning benchmark

In [ ]:
import time, shutil, statistics
from pathlib import Path
import pandas as pd
import pyarrow as pa, pyarrow.compute as pc, pyarrow.dataset as ds, pyarrow.parquet as pq
from google.colab import drive
drive.mount("/content/drive")

W = Path("/content/drive/MyDrive/bft65b_fdic_project")
LOCAL = Path("/content/curated_local"); LOCAL.mkdir(exist_ok=True)
src = LOCAL / "fin_unified.parquet"
shutil.copy(W / "benchmarks/financials_unified_zstd.parquet", src)

# ---------- 1. Dựng bộ dữ liệu phân vùng theo năm ----------
tbl = pq.read_table(src)
year = pc.cast(pc.utf8_slice_codeunits(tbl["REPDTE"], 0, 4), pa.int32())
tbl = tbl.append_column("YEAR", year)

out = LOCAL / "financials_by_year"
if out.exists():
    shutil.rmtree(out)
ds.write_dataset(
    tbl, out, format="parquet",
    partitioning=ds.partitioning(pa.schema([("YEAR", pa.int32())]), flavor="hive"),
    file_options=ds.ParquetFileFormat().make_write_options(compression="zstd"),
)
del tbl

dset = ds.dataset(out, format="parquet", partitioning="hive")
n_rows = dset.count_rows()
print("Số dòng bộ phân vùng:", f"{n_rows:,}", "->", "KHỚP 1,682,738" if n_rows == 1_682_738 else "KHÔNG KHỚP!")
n_files = len(list(out.rglob("*.parquet")))
size_part = sum(f.stat().st_size for f in out.rglob("*.parquet")) / 1024**2
size_single = src.stat().st_size / 1024**2
print(f"File đơn: {size_single:.1f} MB, {pq.ParquetFile(src).num_row_groups} row group")
print(f"Bộ phân vùng: {size_part:.1f} MB, {n_files} file ({n_files} năm)")

# ---------- 2. Đo partition pruning ----------
COLS = ["CERT", "REPDTE", "ASSET", "EQ", "ROA", "NCLNLS"]

def timed(fn, reps=5):
    fn()                                   # lần khởi động, bỏ qua
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); r = fn(); ts.append(time.perf_counter() - t0)
    return statistics.median(ts), r.num_rows

def single(lo, hi):
    if lo is None:
        return lambda: pq.read_table(src, columns=COLS)
    return lambda: pq.read_table(src, columns=COLS,
                                 filters=[("REPDTE", ">=", f"{lo}0101"), ("REPDTE", "<=", f"{hi}1231")])

def parted(lo, hi):
    if lo is None:
        return lambda: dset.to_table(columns=COLS)
    f = (ds.field("YEAR") >= lo) & (ds.field("YEAR") <= hi)
    return lambda: dset.to_table(columns=COLS, filter=f)

rows = []
for label, lo, hi in [("all years (1984-2026)", None, None), ("2008-2012 (5 years)", 2008, 2012), ("2020 only", 2020, 2020)]:
    for layout, mk in [("single file, row-group filter", single), ("partitioned by YEAR", parted)]:
        sec, n = timed(mk(lo, hi))
        rows.append({"scenario": label, "layout": layout, "rows_read": n, "seconds_median": round(sec, 3)})
res = pd.DataFrame(rows)
print("\n=== PARTITION PRUNING (6 cột, đĩa local, trung vị của 5 lần) ===")
print(res.to_string(index=False))

# ---------- 3. Lưu layout chính thức lên Drive + kết quả ----------
dest = W / "data_curated" / "financials_by_year"
dest.parent.mkdir(exist_ok=True)
shutil.copytree(out, dest, dirs_exist_ok=True)
res.to_csv(W / "benchmarks" / "partition_pruning.csv", index=False)
(W / "results").mkdir(exist_ok=True)
res.to_csv(W / "results" / "partition_pruning.csv", index=False)
print("\nĐã lưu layout chính thức:", dest)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Số dòng bộ phân vùng: 1,682,738 -> KHỚP 1,682,738
File đơn: 144.3 MB, 2 row group
Bộ phân vùng: 184.6 MB, 43 file (43 năm)

=== PARTITION PRUNING (6 cột, đĩa local, trung vị của 5 lần) ===
             scenario                        layout  rows_read  seconds_median
all years (1984-2026) single file, row-group filter    1682738           0.248
all years (1984-2026)           partitioned by YEAR    1682738           0.408
  2008-2012 (5 years) single file, row-group filter     157851           0.122
  2008-2012 (5 years)           partitioned by YEAR     157851           0.042
            2020 only single file, row-group filter      20478           0.119
            2020 only           partitioned by YEAR      20478           0.008

Đã lưu layout chính thức: /content/drive/MyDrive/bft65b_fdic_project/data_curated/financials_by_year


## Part 2 - Data quality

In [ ]:
import numpy as np

fin = df_fin.copy()
fails = pd.read_parquet(DATA_DIR / "failures.parquet")
inst = pd.read_parquet(DATA_DIR / "institutions.parquet")

fin["YEAR"] = fin["REPDTE"].astype(str).str[:4].astype(int)
fin["DECADE"] = (fin["YEAR"] // 10) * 10

# 1. Tỷ lệ missing toàn cục và theo thập kỷ
miss_overall = (fin.isna().mean() * 100).round(2).rename("Missing_%_Overall")
miss_by_decade = fin.groupby("DECADE").apply(lambda g: g.isna().mean() * 100).T.round(2)
dq_missing = pd.concat([miss_overall, miss_by_decade], axis=1)
print("=== 1. TỶ LỆ MISSING (%) THEO CỘT & THẬP KỶ ===")
print(dq_missing.to_string())

# 2. Kiểm tra bất thường logic kế toán / giá trị ngoại lai
anomalies = {
    "ASSET <= 0 or NaN": int((fin["ASSET"].isna() | (fin["ASSET"] <= 0)).sum()),
    "DEP < 0": int((fin["DEP"] < 0).sum()),
    "LNLSGR < 0": int((fin["LNLSGR"] < 0).sum()),
    "EQ / ASSET < -1 or > 1": int(((fin["EQ"] / fin["ASSET"] < -1) | (fin["EQ"] / fin["ASSET"] > 1)).sum()),
    "ROA < -50 or > 50 (%)": int(((fin["ROA"] < -50) | (fin["ROA"] > 50)).sum()),
}
print("\n=== 2. CÁC BẤT THƯỜNG GIÁ TRỊ ===")
for k, v in anomalies.items():
    print(f"{k:<30}: {v:,}")

# 3. Kiểm tra đối chiếu khóa CERT giữa 3 bảng & phân loại exits
fin_certs = set(fin["CERT"].unique())
fail_certs = set(fails["CERT"].unique())
inst_certs = set(inst["CERT"].unique())

print("\n=== 3. ĐỐI CHIẾU KHÓA CERT & EXITS ===")
print(f"CERT trong financials nhưng thiếu ở institutions : {len(fin_certs - inst_certs):,}")
print(f"CERT trong failures nhưng thiếu ở financials     : {len(fail_certs - fin_certs):,}")
print(f"Số CERT trùng lặp trong bảng failures            : {fails['CERT'].duplicated().sum():,}")
print("\nPhân bố mã CHANGEC1 (lý do ngừng báo cáo ở institutions, ACTIVE==0):")
print(inst.loc[inst["ACTIVE"] == 0, "CHANGEC1"].value_counts(dropna=False).head(10))

=== 1. TỶ LỆ MISSING (%) THEO CỘT & THẬP KỶ ===
         Missing_%_Overall    1980    1990   2000  2010  2020
EINTEXP               0.38    0.73    0.29   0.22  0.26  0.36
NIMY                  0.00    0.00    0.00   0.00  0.00  0.00
ORE                   0.25    0.18    0.29   0.22  0.26  0.36
LNATRES               0.25    0.18    0.29   0.22  0.26  0.36
REPDTE                0.00    0.00    0.00   0.00  0.00  0.00
ELNATR                2.57    9.49    0.29   0.22  0.26  0.36
STNAME                0.00    0.00    0.00   0.00  0.00  0.00
RBC1AAJ               0.00    0.00    0.00   0.00  0.00  0.00
INTINC                0.28    0.31    0.29   0.22  0.26  0.36
LNCI                  0.00    0.00    0.00   0.00  0.00  0.00
SC                    0.00    0.00    0.00   0.00  0.00  0.00
NTLNLS                4.75   18.25    0.29   0.22  0.26  0.36
NCLNLS                0.15    0.60    0.00   0.00  0.00  0.00
LNRE                  0.00    0.00    0.00   0.00  0.00  0.00
LNLSNET               

/tmp/ipykernel_630/1521120703.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  miss_by_decade = fin.groupby("DECADE").apply(lambda g: g.isna().mean() * 100).T.round(2)


In [ ]:
# 1. Soi 531 dòng trùng CERT trong bảng failures
dup_fails = fails[fails.duplicated("CERT", keep=False)].sort_values(["CERT", "FAILDATE"])
print("=== MẪU CÁC CERT TRÙNG TRONG FAILURES ===")
print(dup_fails.head(10).to_string(index=False))
print("Số cặp trùng cả (CERT, FAILDATE):", fails.duplicated(["CERT", "FAILDATE"]).sum())
print("Khoảng năm FAILDATE trong failures:", fails["FAILYR"].min(), "->", fails["FAILYR"].max())

# 2. Kiểm tra thư viện polars và pyspark trên Colab
import polars as pl
print("\nPolars version:", pl.__version__)

try:
    import pyspark
    print("PySpark version:", pyspark.__version__)
except ImportError:
    print("Chưa cài pyspark -> chạy: !pip install -q pyspark")

=== MẪU CÁC CERT TRÙNG TRONG FAILURES ===
   QBFDEP         COST  QBFASSET   FAILDATE   CERT       CITYST    RESTYPE FAILYR                             NAME
1307005.0     0.000000 1359158.0 10/30/1992 1722.0   DALLAS, TX    FAILURE   1992   FIRST CITY, TEXAS - DALLAS, NA
 489266.0 79114.979658  590295.0  4/20/1988 1722.0   DALLAS, TX ASSISTANCE   1988       FIRST CITY, TEXAS - DALLAS
 488695.0     0.000000  532248.0 10/30/1992 3093.0 BEAUMONT, TX    FAILURE   1992 FIRST CITY, TEXAS - BEAUMONT, NA
 399545.0 12305.419334  447439.0  4/20/1988 3093.0 BEAUMONT, TX ASSISTANCE   1988     FIRST CITY, TEXAS - BEAUMONT
 318838.0     0.000000  340189.0 10/30/1992 3120.0    BRYAN, TX    FAILURE   1992    FIRST CITY, TEXAS - BRYAN, NA
 206807.0  1537.590998  238427.0  4/20/1988 3120.0    BRYAN, TX ASSISTANCE   1988        FIRST CITY, TEXAS - BRYAN
 338207.0 64630.337000  340938.0  1/26/1990 3164.0   DALLAS, TX    FAILURE   1990           BANCTEXAS DALLAS, N.A.
 382732.0 58045.447000  541345.0  7/17

### Part 2 - Automated data-quality report (written to `results/data_quality_report.md`)

In [ ]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

W = Path("/content/drive/MyDrive/bft65b_fdic_project")
R = W / "results"; R.mkdir(exist_ok=True)

def md_table(df, floatfmt="{:,.2f}"):
    df = df.reset_index()
    head = "| " + " | ".join(map(str, df.columns)) + " |"
    sep = "|" + "|".join(["---"] * len(df.columns)) + "|"
    rows = []
    for _, r in df.iterrows():
        cells = [floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in r]
        rows.append("| " + " | ".join(cells) + " |")
    return "\n".join([head, sep] + rows)

# ---------- Đọc dữ liệu ----------
fin = pd.read_parquet(W / "benchmarks/financials_unified_zstd.parquet")
fails_raw = pd.read_parquet(W / "data_raw/fdic/failures.parquet")
inst_raw = pd.read_parquet(W / "data_raw/fdic/institutions.parquet")

fin["YEAR"] = fin["REPDTE"].astype(str).str[:4].astype(int)
fin["Q"] = fin["YEAR"] * 4 + (fin["REPDTE"].astype(str).str[4:6].astype(int) - 1) // 3 + 1
fin["DECADE"] = (fin["YEAR"] // 10) * 10

# ---------- 1. Tổng quan ----------
overview = pd.Series({
    "Rows (bank-quarters)": f"{len(fin):,}",
    "Quarters": f"{fin['REPDTE'].nunique():,}",
    "First / last REPDTE": f"{fin['REPDTE'].min()} / {fin['REPDTE'].max()}",
    "Distinct banks (CERT)": f"{fin['CERT'].nunique():,}",
    "Duplicate (CERT, REPDTE)": f"{int(fin.duplicated(['CERT', 'REPDTE']).sum()):,}",
}, name="value")

# ---------- 2. Missing theo cột và thập kỷ ----------
cols = [c for c in fin.columns if c not in ("YEAR", "Q", "DECADE")]
miss_dec = fin[cols].isna().groupby(fin["DECADE"]).mean().T * 100
miss = pd.concat([(fin[cols].isna().mean() * 100).rename("overall"), miss_dec], axis=1).round(2)
miss.columns = ["overall"] + [f"{int(c)}s" for c in miss_dec.columns]
miss = miss.sort_values("overall", ascending=False)
miss.to_csv(R / "dq_missing_by_decade.csv")

# ---------- 3. Bất thường giá trị ----------
eqa = fin["EQ"] / fin["ASSET"]
anom = pd.Series({
    "ASSET <= 0 or missing": int((fin["ASSET"].isna() | (fin["ASSET"] <= 0)).sum()),
    "DEP < 0": int((fin["DEP"] < 0).sum()),
    "LNLSGR < 0 (set to missing in pipeline)": int((fin["LNLSGR"] < 0).sum()),
    "EQ/ASSET outside [-1, 1] (winsorised in model pipeline)": int(((eqa < -1) | (eqa > 1)).sum()),
    "ROA outside +/-50 (winsorised in model pipeline)": int(((fin["ROA"] < -50) | (fin["ROA"] > 50)).sum()),
}, name="rows")
anom.to_csv(R / "dq_anomalies.csv")

# ---------- 4. Đối chiếu khóa giữa 3 bảng ----------
fails_raw["FAILYR_N"] = pd.to_numeric(fails_raw["FAILYR"], errors="coerce")
fl = fails_raw[(fails_raw["RESTYPE"] == "FAILURE") & (fails_raw["FAILYR_N"] >= 1984)]
fin_certs = set(fin["CERT"].unique())
inst_certs = set(inst_raw["CERT"].dropna().astype(int))
keys = pd.Series({
    "failures: rows with missing CERT (all years / types)": int(fails_raw["CERT"].isna().sum()),
    "failures: FAILURE events since 1984": len(fl),
    "  of which missing CERT": int(fl["CERT"].isna().sum()),
    "  of which CERT not in financials (cannot be in the panel)": int((fl["CERT"].notna() & ~fl["CERT"].isin(fin_certs)).sum()),
    "failures: CERTs with more than one event (FAILURE + ASSISTANCE etc.)": int(fails_raw["CERT"].dropna().duplicated().sum()),
    "financials: banks (CERT) not in institutions": len(fin_certs - inst_certs),
    "financials: rows of those banks": int((~fin["CERT"].isin(inst_certs)).sum()),
}, name="count")
keys.to_csv(R / "dq_key_reconciliation.csv")

# ---------- 5. Lý do ngừng báo cáo (CHANGEC1) so với danh sách phá sản ----------
failed = set(fl["CERT"].dropna().astype(int))
x = inst_raw[inst_raw["ACTIVE"] == 0].copy()
x["is_failed"] = x["CERT"].isin(failed)
exit_tab = x.groupby("CHANGEC1")["is_failed"].agg(n_inactive="size", n_failed="sum")
exit_tab["pct_failed"] = (100 * exit_tab["n_failed"] / exit_tab["n_inactive"]).round(1)
exit_tab = exit_tab.sort_values("n_inactive", ascending=False)
exit_tab.to_csv(R / "dq_exit_codes_vs_failures.csv")

# ---------- 6. Ngân hàng bị thiếu quý giữa chừng ----------
s = fin[["CERT", "Q"]].sort_values(["CERT", "Q"])
d = s.groupby("CERT")["Q"].diff()
gap = d[d > 1]
gaps = pd.Series({
    "banks with at least one gap": int(s.loc[gap.index, "CERT"].nunique()),
    "number of gaps": int(len(gap)),
    "total missing quarters inside gaps": int((gap - 1).sum()),
}, name="count")
gaps.to_csv(R / "dq_quarter_gaps.csv")

# ---------- Viết báo cáo Markdown ----------
report = f"""# Data quality report - FDIC Call Report panel
Generated {time.strftime('%Y-%m-%d %H:%M')} by an automated script (cell "Part 2").

## 1. Overview
{md_table(overview.to_frame())}

## 2. Missing values (% of rows) by column and decade
{md_table(miss)}

## 3. Value anomalies
{md_table(anom.to_frame())}

## 4. Key reconciliation across the three tables
{md_table(keys.to_frame())}

## 5. Why banks stop reporting (CHANGEC1, ACTIVE == 0) versus the failure list
`n_failed` counts banks in the FAILURE list (since 1984). Codes that mix failures and other exits are the reason
the failure label is built from the failures table, never inferred from CHANGEC1.
{md_table(exit_tab.head(12))}

## 6. Banks with missing quarters inside their history
Lag features (shift(4), pct_change(4)) assume consecutive quarters.
{md_table(gaps.to_frame())}

## 7. Treatment decisions
| Issue | Treatment |
|---|---|
| LNLSGR < 0 | Set to missing before ratios are computed |
| Failures table: rows without CERT; events before 1984 | Dropped (no FAILURE since 1984 is lost for lack of CERT, see section 4) |
| Failures table: duplicate (CERT, FAILDATE) | Merged; only RESTYPE == FAILURE defines the label (assistance and mergers are not failures) |
| FAILURE events whose CERT has no Call Report | Cannot enter the panel; counted in section 4 and reported as a limitation |
| P3ASSET (heavily missing before 2000) | Not used as a feature |
| Extreme ROA / EQ-to-ASSET values | Winsorised (0.5%-99.5%) inside the model pipeline, fitted on training data only |
| Banks absent from institutions table | Treated as continuing (no end date) |
| Banks with missing quarters | Reported in section 6; lag features may span more than 4 quarters for these banks |
"""
(R / "data_quality_report.md").write_text(report, encoding="utf-8")

print(report)
print("\nĐã lưu vào:", R)

Mounted at /content/drive
# Data quality report - FDIC Call Report panel
Generated 2026-11-21 03:35 by an automated script (cell "Part 2").

## 1. Overview
| index | value |
|---|---|
| Rows (bank-quarters) | 1,682,738 |
| Quarters | 170 |
| First / last REPDTE | 19840331 / 20260630 |
| Distinct banks (CERT) | 24,064 |
| Duplicate (CERT, REPDTE) | 0 |

## 2. Missing values (% of rows) by column and decade
| index | overall | 1980s | 1990s | 2000s | 2010s | 2020s |
|---|---|---|---|---|---|---|
| P3ASSET | 57.77 | 100.00 | 100.00 | 11.12 | 0.00 | 0.00 |
| NTLNLS | 4.75 | 18.25 | 0.29 | 0.22 | 0.26 | 0.36 |
| NAASSET | 4.62 | 18.53 | 0.00 | 0.00 | 0.00 | 0.00 |
| P9ASSET | 4.62 | 18.53 | 0.00 | 0.00 | 0.00 | 0.00 |
| ELNATR | 2.57 | 9.49 | 0.29 | 0.22 | 0.26 | 0.36 |
| ROE | 1.11 | 2.97 | 0.81 | 0.22 | 0.27 | 0.47 |
| BRO | 1.00 | 3.19 | 0.29 | 0.22 | 0.26 | 0.36 |
| EINTEXP | 0.38 | 0.73 | 0.29 | 0.22 | 0.26 | 0.36 |
| INTINC | 0.28 | 0.31 | 0.29 | 0.22 | 0.26 | 0.36 |
| NONIX | 0.28 | 

### Part 2 - Exit codes (CHANGEC1) versus the failure list

In [ ]:
import pandas as pd
from pathlib import Path
W = Path("/content/drive/MyDrive/bft65b_fdic_project")

raw_f = pd.read_parquet(W / "data_raw/fdic/failures.parquet")
inst = pd.read_parquet(W / "data_raw/fdic/institutions.parquet")
fin_certs = set(pd.read_parquet(W / "benchmarks/financials_unified_zstd.parquet", columns=["CERT"])["CERT"])

raw_f["FAILYR_N"] = pd.to_numeric(raw_f["FAILYR"], errors="coerce")
post = raw_f[(raw_f.FAILYR_N >= 1984) & (raw_f.RESTYPE == "FAILURE")]
print("Vụ FAILURE từ 1984:", len(post))
print("  thiếu CERT:", int(post.CERT.isna().sum()))
print(post[post.CERT.isna()].groupby("FAILYR_N").size().to_string())
print("  có CERT nhưng không có trong financials:",
      int((post.CERT.notna() & ~post.CERT.isin(fin_certs)).sum()))

failed = set(post.CERT.dropna().astype(int))
x = inst[inst.ACTIVE == 0].copy()
x["is_failed"] = x.CERT.isin(failed)
print("\nCHANGEC1 của các ngân hàng ngừng hoạt động (size = tổng, sum = số vụ phá sản):")
print(x.groupby("CHANGEC1").is_failed.agg(["size", "sum"]).sort_values("size", ascending=False).head(10))

Vụ FAILURE từ 1984: 2863
  thiếu CERT: 0
Series([], )
  có CERT nhưng không có trong financials: 21

CHANGEC1 của các ngân hàng ngừng hoạt động (size = tổng, sum = số vụ phá sản):
           size   sum
CHANGEC1             
223.0     17703     0
211.0      2538  2392
221.0       816     1
217.0       666   159
240.0       575     0
213.0       567    96
230.0       236   184
224.0       191     0
222.0       173     0
235.0       110    25


## Part 3 - Processing at scale (Polars lazy pipeline, pandas, Spark)

In [ ]:
import time
import tracemalloc
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl

WORK_DIR = Path("/content/drive/MyDrive/bft65b_fdic_project")
DATA_DIR = WORK_DIR / "data_raw" / "fdic"
BENCH_DIR = WORK_DIR / "benchmarks"
PROC_DIR = WORK_DIR / "data_processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

pq_unified_path = BENCH_DIR / "financials_unified_zstd.parquet"

# 1. Làm sạch bảng failures (loại dòng thiếu CERT, gộp 70 cặp trùng CERT-FAILDATE)
fails_raw = pd.read_parquet(DATA_DIR / "failures.parquet")
print("Số dòng failures bị thiếu CERT (NaN):", int(fails_raw["CERT"].isna().sum()))

fails_raw = fails_raw.dropna(subset=["CERT"]).copy()
fails_raw["CERT"] = fails_raw["CERT"].astype("int64")
fails_raw["FAIL_DT"] = pd.to_datetime(fails_raw["FAILDATE"])
fails_clean = (
    fails_raw.sort_values(["CERT", "FAIL_DT", "RESTYPE"])
    .groupby(["CERT", "FAIL_DT"], as_index=False)
    .agg(
        FAILYR=("FAILYR", "first"),
        RESTYPE=("RESTYPE", lambda s: "FAILURE" if "FAILURE" in set(s) else s.iloc[0]),
        COST=("COST", "sum"),
        QBFASSET=("QBFASSET", "max"),
        QBFDEP=("QBFDEP", "max"),
    )
)
fails_clean = fails_clean[fails_clean["FAIL_DT"] >= "1984-01-01"].reset_index(drop=True)
fails_clean.to_parquet(PROC_DIR / "failures_clean.parquet", index=False)
print("Số sự kiện failure/assistance sạch từ 1984:", len(fails_clean))

# 2. Làm sạch bảng institutions (phân loại lý do ngừng báo cáo point-in-time)
inst_raw = pd.read_parquet(DATA_DIR / "institutions.parquet")
inst_clean = inst_raw.dropna(subset=["CERT"]).drop_duplicates("CERT", keep="last").copy()
inst_clean["CERT"] = inst_clean["CERT"].astype("int64")
inst_clean["END_DT"] = pd.to_datetime(inst_clean["ENDEFYMD"].astype(str).str.replace(r"\.0$", "", regex=True), format="%Y%m%d", errors="coerce")
inst_clean["EST_DT"] = pd.to_datetime(inst_clean["ESTYMD"].astype(str).str.replace(r"\.0$", "", regex=True), format="%Y%m%d", errors="coerce")
inst_clean["IS_MERGER_EXIT"] = (inst_clean["ACTIVE"] == 0) & (inst_clean["CHANGEC1"] == 223)
inst_clean.to_parquet(PROC_DIR / "institutions_clean.parquet", index=False,
                      coerce_timestamps="us", allow_truncated_timestamps=True)
# 3. Xây dựng Polars Lazy Pipeline cho bảng financials (Part 3)
def build_polars_lazy_plan(parquet_path: Path, inst_path: Path) -> pl.LazyFrame:
    lf_fin = pl.scan_parquet(parquet_path)
    lf_inst = pl.scan_parquet(inst_path).select(["CERT", "BKCLASS", "ACTIVE", "CHANGEC1", "END_DT", "EST_DT", "IS_MERGER_EXIT"])

    lf = (
        lf_fin
        .with_columns([
            pl.col("REPDTE").str.strptime(pl.Date, "%Y%m%d").alias("REP_DT"),
            pl.when(pl.col("LNLSGR") < 0).then(None).otherwise(pl.col("LNLSGR")).alias("LNLSGR"),
        ])
        .join(lf_inst, on="CERT", how="left")
        .filter(pl.col("END_DT").is_null() | (pl.col("REP_DT") <= pl.col("END_DT")))
        .sort(["CERT", "REP_DT"])
        .with_columns([
            (pl.col("EQ") / pl.col("ASSET")).alias("EQ_TO_ASSET"),
            (pl.col("NCLNLS") / pl.col("LNLSGR").clip(lower_bound=1.0)).alias("NPL_RATIO"),
            (pl.col("LNATRES") / pl.col("LNLSGR").clip(lower_bound=1.0)).alias("RESERVE_TO_LOANS"),
            ((pl.col("CHBAL") + pl.col("SC")) / pl.col("ASSET")).alias("LIQ_RATIO"),
            (pl.col("BRO") / pl.col("DEP").clip(lower_bound=1.0)).alias("BROKERED_DEP_RATIO"),
            (pl.col("LNRE") / pl.col("LNLSGR").clip(lower_bound=1.0)).alias("RE_LOAN_SHARE"),
            pl.col("ASSET").log().alias("LOG_ASSET"),
        ])
        .with_columns([
            (pl.col("EQ_TO_ASSET") - pl.col("EQ_TO_ASSET").shift(1).over("CERT")).alias("D_EQ_TO_ASSET_1Q"),
            (pl.col("EQ_TO_ASSET") - pl.col("EQ_TO_ASSET").shift(4).over("CERT")).alias("D_EQ_TO_ASSET_4Q"),
            (pl.col("NPL_RATIO") - pl.col("NPL_RATIO").shift(4).over("CERT")).alias("D_NPL_4Q"),
            (pl.col("ROA") - pl.col("ROA").shift(4).over("CERT")).alias("D_ROA_4Q"),
            pl.col("ROA").rolling_std(window_size=4, min_samples=2).over("CERT").alias("VOL_ROA_4Q"),
            pl.col("ASSET").pct_change(n=4).over("CERT").alias("ASSET_GROWTH_4Q"),
            pl.when(pl.col("REP_DT") == pl.col("REP_DT").max().over("CERT"))
              .then(
                  pl.when(pl.col("IS_MERGER_EXIT") == True).then(pl.lit("MERGER_EXIT"))
                    .when(pl.col("ACTIVE") == 0).then(pl.lit("NON_MERGER_EXIT"))
                    .otherwise(pl.lit("ACTIVE"))
              )
              .otherwise(pl.lit("CONTINUING"))
              .alias("PANEL_STATUS"),
        ])
    )
    return lf

lf_plan = build_polars_lazy_plan(pq_unified_path, PROC_DIR / "institutions_clean.parquet")
print("\n=== POLARS OPTIMIZED QUERY PLAN (PART 3) ===")
print(lf_plan.explain(optimized=True))

Số dòng failures bị thiếu CERT (NaN): 488
Số sự kiện failure/assistance sạch từ 1984: 3275

=== POLARS OPTIMIZED QUERY PLAN (PART 3) ===
simple π 54/54 ["EINTEXP", "NIMY", "ORE", ... 51 other columns]
   WITH_COLUMNS:
   [[(col("EQ_TO_ASSET")) - (col("EQ_TO_ASSET").shift([dyn int: 1]).over([col("CERT")]))].alias("D_EQ_TO_ASSET_1Q"), [(col("EQ_TO_ASSET")) - (col("EQ_TO_ASSET").shift([dyn int: 4]).over([col("CERT")]))].alias("D_EQ_TO_ASSET_4Q"), [(col("NPL_RATIO")) - (col("NPL_RATIO").shift([dyn int: 4]).over([col("CERT")]))].alias("D_NPL_4Q")] 
    simple π 51/51 ["EINTEXP", "NIMY", "ORE", ... 48 other columns]
       WITH_COLUMNS:
       [[(col("EQ")) / (col("ASSET"))].alias("EQ_TO_ASSET"), [(col("NCLNLS")) / (col("__POLARS_CSER_0xf9e39d3780b9f3e1"))].alias("NPL_RATIO"), [(col("LNATRES")) / (col("__POLARS_CSER_0xf9e39d3780b9f3e1"))].alias("RESERVE_TO_LOANS"), [([(col("CHBAL")) + (col("SC"))]) / (col("ASSET"))].alias("LIQ_RATIO"), [(col("BRO")) / (col("DEP").clip_min([dyn float: 1]))].a

In [ ]:
import time
panel_clean_path = PROC_DIR / "panel_features_base.parquet"
if panel_clean_path.exists():
    print("panel_features_base.parquet đã có, giữ nguyên:", panel_clean_path)
else:
    t0 = time.perf_counter()
    df_pl = lf_plan.collect()
    df_pl.write_parquet(panel_clean_path, compression="zstd")
    print(f"Đã tạo panel: {len(df_pl):,} dòng trong {time.perf_counter() - t0:.1f}s")

panel_features_base.parquet đã có, giữ nguyên: /content/drive/MyDrive/bft65b_fdic_project/data_processed/panel_features_base.parquet


### Part 3 - Polars vs pandas benchmark and scaling experiment (x1, x2, x4)

In [ ]:
import subprocess, sys, shutil, json
from pathlib import Path
import numpy as np, pandas as pd, polars as pl

WORK_DIR = Path("/content/drive/MyDrive/bft65b_fdic_project")
BENCH_DIR, PROC_DIR = WORK_DIR / "benchmarks", WORK_DIR / "data_processed"
LOCAL = Path("/content/bench_local"); LOCAL.mkdir(exist_ok=True)

shutil.copy(BENCH_DIR / "financials_unified_zstd.parquet", LOCAL / "fin_x1.parquet")
shutil.copy(PROC_DIR / "institutions_clean.parquet", LOCAL / "inst_x1.parquet")

PIPE = r'''
import sys, time, json, resource
engine, fin_path, inst_path, out_path = sys.argv[1:5]
t0 = time.perf_counter()
INST_COLS = ["CERT", "BKCLASS", "ACTIVE", "CHANGEC1", "END_DT", "EST_DT", "IS_MERGER_EXIT"]

if engine == "polars":
    import polars as pl
    lf = (
        pl.scan_parquet(fin_path)
        .with_columns([
            pl.col("REPDTE").str.strptime(pl.Date, "%Y%m%d").alias("REP_DT"),
            pl.when(pl.col("LNLSGR") < 0).then(None).otherwise(pl.col("LNLSGR")).alias("LNLSGR"),
        ])
        .join(pl.scan_parquet(inst_path).select(INST_COLS), on="CERT", how="left")
        .filter(pl.col("END_DT").is_null() | (pl.col("REP_DT") <= pl.col("END_DT")))
        .sort(["CERT", "REP_DT"])
        .with_columns([
            (pl.col("EQ") / pl.col("ASSET")).alias("EQ_TO_ASSET"),
            (pl.col("NCLNLS") / pl.col("LNLSGR").clip(lower_bound=1.0)).alias("NPL_RATIO"),
            (pl.col("LNATRES") / pl.col("LNLSGR").clip(lower_bound=1.0)).alias("RESERVE_TO_LOANS"),
            ((pl.col("CHBAL") + pl.col("SC")) / pl.col("ASSET")).alias("LIQ_RATIO"),
            (pl.col("BRO") / pl.col("DEP").clip(lower_bound=1.0)).alias("BROKERED_DEP_RATIO"),
            (pl.col("LNRE") / pl.col("LNLSGR").clip(lower_bound=1.0)).alias("RE_LOAN_SHARE"),
            pl.col("ASSET").log().alias("LOG_ASSET"),
        ])
        .with_columns([
            (pl.col("EQ_TO_ASSET") - pl.col("EQ_TO_ASSET").shift(1).over("CERT")).alias("D_EQ_TO_ASSET_1Q"),
            (pl.col("EQ_TO_ASSET") - pl.col("EQ_TO_ASSET").shift(4).over("CERT")).alias("D_EQ_TO_ASSET_4Q"),
            (pl.col("NPL_RATIO") - pl.col("NPL_RATIO").shift(4).over("CERT")).alias("D_NPL_4Q"),
            (pl.col("ROA") - pl.col("ROA").shift(4).over("CERT")).alias("D_ROA_4Q"),
            pl.col("ROA").rolling_std(window_size=4, min_samples=2).over("CERT").alias("VOL_ROA_4Q"),
            pl.col("ASSET").pct_change(n=4).over("CERT").alias("ASSET_GROWTH_4Q"),
        ])
    )
    lf.collect().write_parquet(out_path, compression="zstd")
else:
    import numpy as np, pandas as pd
    df = pd.read_parquet(fin_path)
    inst = pd.read_parquet(inst_path)[INST_COLS]
    df["REP_DT"] = pd.to_datetime(df["REPDTE"], format="%Y%m%d")
    df.loc[df["LNLSGR"] < 0, "LNLSGR"] = np.nan
    df = df.merge(inst, on="CERT", how="left")
    df = df[df["END_DT"].isna() | (df["REP_DT"] <= df["END_DT"])].sort_values(["CERT", "REP_DT"]).reset_index(drop=True)
    loans, dep = df["LNLSGR"].clip(lower=1.0), df["DEP"].clip(lower=1.0)
    df["EQ_TO_ASSET"] = df["EQ"] / df["ASSET"]
    df["NPL_RATIO"] = df["NCLNLS"] / loans
    df["RESERVE_TO_LOANS"] = df["LNATRES"] / loans
    df["LIQ_RATIO"] = (df["CHBAL"] + df["SC"]) / df["ASSET"]
    df["BROKERED_DEP_RATIO"] = df["BRO"] / dep
    df["RE_LOAN_SHARE"] = df["LNRE"] / loans
    df["LOG_ASSET"] = np.log(df["ASSET"])
    g = df.groupby("CERT", sort=False)
    df["D_EQ_TO_ASSET_1Q"] = df["EQ_TO_ASSET"] - g["EQ_TO_ASSET"].shift(1)
    df["D_EQ_TO_ASSET_4Q"] = df["EQ_TO_ASSET"] - g["EQ_TO_ASSET"].shift(4)
    df["D_NPL_4Q"] = df["NPL_RATIO"] - g["NPL_RATIO"].shift(4)
    df["D_ROA_4Q"] = df["ROA"] - g["ROA"].shift(4)
    df["VOL_ROA_4Q"] = g["ROA"].transform(lambda s: s.rolling(4, min_periods=2).std())
    df["ASSET_GROWTH_4Q"] = g["ASSET"].pct_change(4, fill_method=None)
    df.to_parquet(out_path, index=False, compression="zstd")

print(json.dumps({"sec": time.perf_counter() - t0,
                  "peak_mb": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024}))
'''
(LOCAL / "pipe.py").write_text(PIPE)

def make_scaled(k):
    for name in ("fin", "inst"):
        d = pl.read_parquet(LOCAL / f"{name}_x1.parquet")
        pl.concat([d.with_columns(pl.col("CERT") + i * 10_000_000) for i in range(k)]) \
          .write_parquet(LOCAL / f"{name}_x{k}.parquet", compression="zstd")
        del d

def run(engine, k, tag):
    out = LOCAL / f"out_{engine}_x{k}_{tag}.parquet"
    r = subprocess.run([sys.executable, str(LOCAL / "pipe.py"), engine,
                        str(LOCAL / f"fin_x{k}.parquet"), str(LOCAL / f"inst_x{k}.parquet"), str(out)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        return {"engine": engine, "scale": k, "rep": tag, "status": f"FAILED rc={r.returncode}"}
    d = json.loads(r.stdout.strip().splitlines()[-1])
    nrows = pl.scan_parquet(out).select(pl.len()).collect().item()
    if not (k == 1 and tag == 0):
        out.unlink(missing_ok=True)
    return {"engine": engine, "scale": k, "rep": tag, "status": "ok",
            "rows": nrows, "sec": d["sec"], "peak_mb": d["peak_mb"]}

results = []
def save():
    pd.DataFrame(results).to_csv(BENCH_DIR / "polars_vs_pandas_scaling.csv", index=False)

# --- x1: 3 lần, xen kẽ hai engine ---
for rep in range(3):
    for eng in ("polars", "pandas"):
        results.append(run(eng, 1, rep)); save(); print(results[-1])

# --- Kiểm tra pandas khớp Polars (từng cột, có dung sai) ---
a = pl.read_parquet(LOCAL / "out_polars_x1_0.parquet").sort(["CERT", "REP_DT"])
b = pl.read_parquet(LOCAL / "out_pandas_x1_0.parquet").sort(["CERT", "REP_DT"])
cols = ["EQ_TO_ASSET", "NPL_RATIO", "RESERVE_TO_LOANS", "LIQ_RATIO", "BROKERED_DEP_RATIO", "RE_LOAN_SHARE",
        "LOG_ASSET", "D_EQ_TO_ASSET_1Q", "D_EQ_TO_ASSET_4Q", "D_NPL_4Q", "D_ROA_4Q", "VOL_ROA_4Q", "ASSET_GROWTH_4Q"]
chk = [{"column": c, "match": bool(np.allclose(a[c].to_numpy(), b[c].to_numpy(), rtol=1e-6, atol=1e-9, equal_nan=True))}
       for c in cols]
print("\nSố dòng Polars/pandas:", a.height, b.height)
print(pd.DataFrame(chk).to_string(index=False))
print("ALL COLUMNS MATCH:", all(x["match"] for x in chk) and a.height == b.height)

# --- Thí nghiệm co giãn x2, x4 (1 lần mỗi mức) ---
for k in (2, 4):
    make_scaled(k)
    for eng in ("polars", "pandas"):
        results.append(run(eng, k, 0)); save(); print(results[-1])
    for name in ("fin", "inst"):
        (LOCAL / f"{name}_x{k}.parquet").unlink(missing_ok=True)

res = pd.DataFrame(results)
ok = res[res.status == "ok"]
summary = ok.groupby(["scale", "engine"]).agg(rows=("rows", "first"), sec_median=("sec", "median"),
                                              sec_min=("sec", "min"), sec_max=("sec", "max"),
                                              peak_mb_median=("peak_mb", "median")).round(1)
print("\n=== BENCHMARK (đĩa local, mỗi engine một tiến trình riêng) ===")
print(summary)
print("\nCác lần chạy thất bại (nếu có):")
print(res[res.status != "ok"].to_string(index=False) if (res.status != "ok").any() else "không có")

{'engine': 'polars', 'scale': 1, 'rep': 0, 'status': 'ok', 'rows': 1682738, 'sec': 8.692732324999724, 'peak_mb': 5824.7109375}
{'engine': 'pandas', 'scale': 1, 'rep': 0, 'status': 'ok', 'rows': 1682738, 'sec': 30.76485965000029, 'peak_mb': 5824.7109375}
{'engine': 'polars', 'scale': 1, 'rep': 1, 'status': 'ok', 'rows': 1682738, 'sec': 11.180460956999923, 'peak_mb': 5824.7109375}
{'engine': 'pandas', 'scale': 1, 'rep': 1, 'status': 'ok', 'rows': 1682738, 'sec': 20.398559105999993, 'peak_mb': 5824.7109375}
{'engine': 'polars', 'scale': 1, 'rep': 2, 'status': 'ok', 'rows': 1682738, 'sec': 9.14011609299996, 'peak_mb': 5824.7109375}
{'engine': 'pandas', 'scale': 1, 'rep': 2, 'status': 'ok', 'rows': 1682738, 'sec': 22.157613698000205, 'peak_mb': 5824.7109375}

Số dòng Polars/pandas: 1682738 1682738
            column  match
       EQ_TO_ASSET   True
         NPL_RATIO   True
  RESERVE_TO_LOANS   True
         LIQ_RATIO   True
BROKERED_DEP_RATIO   True
     RE_LOAN_SHARE   True
         LOG_A

> Note: the `peak_mb` column printed by the cell above is NOT valid. It inherits the peak memory of the parent notebook, so both engines show the same number. Valid per-engine memory measurements are in the memory-measurement cell below. The `VOL_ROA_4Q` mismatch is explained in the column-check cell below (max difference 6e-7, floating-point noise in Polars `rolling_std`).

### Part 3 - Column-by-column check of pandas against Polars (VOL_ROA_4Q)

In [ ]:
import polars as pl, numpy as np
L = "/content/bench_local"
a = pl.read_parquet(f"{L}/out_polars_x1_0.parquet").sort(["CERT", "REP_DT"])
b = pl.read_parquet(f"{L}/out_pandas_x1_0.parquet").sort(["CERT", "REP_DT"])
x = a["VOL_ROA_4Q"].to_numpy().astype(float)
y = b["VOL_ROA_4Q"].to_numpy().astype(float)
nx, ny = np.isnan(x), np.isnan(y)
print("rỗng Polars:", int(nx.sum()), "| rỗng pandas:", int(ny.sum()), "| khác mẫu rỗng:", int((nx != ny).sum()))
both = ~nx & ~ny
d = np.abs(x[both] - y[both])
print("số dòng so sánh được:", int(both.sum()))
print("max |chênh|:", float(d.max()))
for tol in (1e-9, 1e-6, 1e-3):
    print(f"số dòng lệch > {tol}:", int((d > tol).sum()))
idx = np.where(both)[0][np.argsort(-d)[:5]]
print(a.select(["CERT", "REP_DT", "ROA", "VOL_ROA_4Q"])[idx.tolist()])
print("giá trị pandas tương ứng:", y[idx])

rỗng Polars: 24064 | rỗng pandas: 24064 | khác mẫu rỗng: 0
số dòng so sánh được: 1658674
max |chênh|: 6.000069328364332e-07
số dòng lệch > 1e-09: 63
số dòng lệch > 1e-06: 0
số dòng lệch > 0.001: 0
shape: (5, 4)
┌───────┬────────────┬─────┬────────────┐
│ CERT  ┆ REP_DT     ┆ ROA ┆ VOL_ROA_4Q │
│ ---   ┆ ---        ┆ --- ┆ ---        │
│ i64   ┆ date       ┆ f64 ┆ f64        │
╞═══════╪════════════╪═════╪════════════╡
│ 27229 ┆ 2000-03-31 ┆ 0.0 ┆ 6.0001e-7  │
│ 27229 ┆ 2000-12-31 ┆ 0.0 ┆ 6.0001e-7  │
│ 27229 ┆ 2001-03-31 ┆ 0.0 ┆ 6.0001e-7  │
│ 27229 ┆ 2000-06-30 ┆ 0.0 ┆ 6.0001e-7  │
│ 27229 ┆ 2000-09-30 ┆ 0.0 ┆ 6.0001e-7  │
└───────┴────────────┴─────┴────────────┘
giá trị pandas tương ứng: [0. 0. 0. 0. 0.]


### Part 3 - Per-engine peak memory and scaling (measured per child process)

In [ ]:
import subprocess, sys, json, time
from pathlib import Path
import psutil, pandas as pd
from google.colab import drive
drive.mount("/content/drive")

LOCAL = Path("/content/bench_local")
BENCH_DIR = Path("/content/drive/MyDrive/bft65b_fdic_project/benchmarks")
assert (LOCAL / "pipe.py").exists() and (LOCAL / "fin_x1.parquet").exists(), \
    "Thiếu file trong /content/bench_local. Hãy chạy lại ô benchmark cũ (bước 4) trước."

vm = psutil.virtual_memory()
print(f"RAM máy: tổng {vm.total/1e9:.1f} GB | còn trống {vm.available/1e9:.1f} GB")

MAKE = r'''
import sys, polars as pl
k, L = int(sys.argv[1]), sys.argv[2]
for name in ("fin", "inst"):
    d = pl.read_parquet(f"{L}/{name}_x1.parquet")
    pl.concat([d.with_columns(pl.col("CERT") + i * 10_000_000) for i in range(k)]) \
      .write_parquet(f"{L}/{name}_x{k}.parquet", compression="zstd")
'''
(LOCAL / "make_scaled.py").write_text(MAKE)

def run(engine, k, tag):
    out = LOCAL / f"out_{engine}_x{k}_{tag}.parquet"
    p = subprocess.Popen([sys.executable, str(LOCAL / "pipe.py"), engine,
                          str(LOCAL / f"fin_x{k}.parquet"), str(LOCAL / f"inst_x{k}.parquet"), str(out)],
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    proc, peak = psutil.Process(p.pid), 0
    while p.poll() is None:
        try:
            rss = proc.memory_info().rss + sum(c.memory_info().rss for c in proc.children(recursive=True))
            peak = max(peak, rss)
        except psutil.Error:
            pass
        time.sleep(0.2)
    stdout, _ = p.communicate()
    row = {"engine": engine, "scale": k, "rep": tag, "peak_gb": round(peak / 1e9, 2)}
    if p.returncode != 0:
        return {**row, "status": f"FAILED rc={p.returncode}"}
    row.update(status="ok", sec=json.loads(stdout.strip().splitlines()[-1])["sec"])
    out.unlink(missing_ok=True)
    return row

results = []
def save():
    pd.DataFrame(results).to_csv(BENCH_DIR / "polars_vs_pandas_scaling_v2.csv", index=False)

plan = {1: 3, 2: 2, 4: 1}   # mức co giãn: số lần lặp
for k, reps in plan.items():
    if k > 1:
        subprocess.run([sys.executable, str(LOCAL / "make_scaled.py"), str(k), str(LOCAL)], check=True)
    for rep in range(reps):
        for eng in ("polars", "pandas"):
            results.append(run(eng, k, rep)); save(); print(results[-1])
    if k > 1:
        for name in ("fin", "inst"):
            (LOCAL / f"{name}_x{k}.parquet").unlink(missing_ok=True)

res = pd.DataFrame(results)
ok = res[res.status == "ok"]
print("\n=== BENCHMARK (RAM đo riêng từng tiến trình con) ===")
print(ok.groupby(["scale", "engine"]).agg(sec_median=("sec", "median"), sec_min=("sec", "min"),
                                          sec_max=("sec", "max"), peak_gb_max=("peak_gb", "max")).round(2))
print("\nLần thất bại (peak_gb = RAM đạt tới trước khi bị hệ điều hành giết):")
print(res[res.status != "ok"][["engine", "scale", "status", "peak_gb"]].to_string(index=False)
      if (res.status != "ok").any() else "không có")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAM máy: tổng 13.6 GB | còn trống 12.0 GB
{'engine': 'polars', 'scale': 1, 'rep': 0, 'peak_gb': 1.52, 'status': 'ok', 'sec': 25.046579688999827}
{'engine': 'pandas', 'scale': 1, 'rep': 0, 'peak_gb': 2.58, 'status': 'ok', 'sec': 22.220542395999928}
{'engine': 'polars', 'scale': 1, 'rep': 1, 'peak_gb': 1.52, 'status': 'ok', 'sec': 9.919486337999842}
{'engine': 'pandas', 'scale': 1, 'rep': 1, 'peak_gb': 2.64, 'status': 'ok', 'sec': 20.741108216000157}
{'engine': 'polars', 'scale': 1, 'rep': 2, 'peak_gb': 1.52, 'status': 'ok', 'sec': 11.674566952000077}
{'engine': 'pandas', 'scale': 1, 'rep': 2, 'peak_gb': 2.57, 'status': 'ok', 'sec': 20.268236116000026}
{'engine': 'polars', 'scale': 2, 'rep': 0, 'peak_gb': 3.37, 'status': 'ok', 'sec': 19.373737966000135}
{'engine': 'pandas', 'scale': 2, 'rep': 0, 'peak_gb': 5.1, 'status': 'ok', 'sec': 40.931970822999574}
{'engin

### Part 3 - PySpark reproduction and equivalence check with Polars

In [ ]:
import time, math
import pandas as pd
import polars as pl
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

WORK_DIR = Path("/content/drive/MyDrive/bft65b_fdic_project")
BENCH_DIR = WORK_DIR / "benchmarks"
PROC_DIR = WORK_DIR / "data_processed"
pq_unified_path = BENCH_DIR / "financials_unified_zstd.parquet"

# 1. Ghi lại institutions_clean với timestamp microseconds để Spark đọc được
inst_fix = pd.read_parquet(PROC_DIR / "institutions_clean.parquet")
inst_fix.to_parquet(PROC_DIR / "institutions_clean.parquet", index=False,
                    coerce_timestamps="us", allow_truncated_timestamps=True)

# 2. Đọc kết quả Polars đã lưu
df_pl = pl.read_parquet(PROC_DIR / "panel_features_base.parquet")

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("BFT65B_FDIC_Pipeline")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

t0 = time.perf_counter()
sp_fin = spark.read.parquet(str(pq_unified_path))
sp_inst = spark.read.parquet(str(PROC_DIR / "institutions_clean.parquet")).select(
    "CERT", F.to_date("END_DT").alias("END_DT"))

w_cert = Window.partitionBy("CERT").orderBy("REP_DT")
sp_out = (
    sp_fin
    .withColumn("REP_DT", F.to_date(F.col("REPDTE"), "yyyyMMdd"))
    .withColumn("LNLSGR", F.when(F.col("LNLSGR") < 0, None).otherwise(F.col("LNLSGR")))
    .join(sp_inst, on="CERT", how="left")
    .filter(F.col("END_DT").isNull() | (F.col("REP_DT") <= F.col("END_DT")))
    .withColumn("EQ_TO_ASSET", F.col("EQ") / F.col("ASSET"))
    # SỬA Ở ĐÂY: giữ null khi LNLSGR null (greatest của Spark bỏ qua null)
    .withColumn(
        "NPL_RATIO",
        F.when(F.col("LNLSGR").isNull(), F.lit(None).cast("double"))
         .otherwise(F.col("NCLNLS") / F.greatest(F.col("LNLSGR"), F.lit(1.0)))
    )
    .withColumn("D_EQ_TO_ASSET_4Q", F.col("EQ_TO_ASSET") - F.lag("EQ_TO_ASSET", 4).over(w_cert))
)

cols_check = ["EQ_TO_ASSET", "NPL_RATIO", "D_EQ_TO_ASSET_4Q"]
aggs = [F.count("*").alias("n_rows")]
for c in cols_check:
    aggs += [F.avg(c).alias(f"mean_{c}"),
             F.sum(F.col(c).isNull().cast("int")).alias(f"nulls_{c}")]

sp_summary = sp_out.agg(*aggs).collect()[0]
t_spark = time.perf_counter() - t0

rows = [{"metric": "n_rows", "polars": len(df_pl), "spark": sp_summary["n_rows"],
         "match": len(df_pl) == sp_summary["n_rows"]}]
for c in cols_check:
    pm, sm = df_pl[c].mean(), sp_summary[f"mean_{c}"]
    rows.append({"metric": f"mean {c}", "polars": pm, "spark": sm,
                 "match": math.isclose(pm, sm, rel_tol=1e-9, abs_tol=1e-12)})
    pn, sn = df_pl[c].null_count(), sp_summary[f"nulls_{c}"]
    rows.append({"metric": f"nulls {c}", "polars": pn, "spark": sn, "match": pn == sn})

cmp_df = pd.DataFrame(rows)
print(f"PySpark local[*] runtime: {t_spark:.2f}s")
print(cmp_df.to_string(index=False))
print("\nALL MATCH:", bool(cmp_df["match"].all()))
cmp_df.to_csv(BENCH_DIR / "polars_vs_spark_check.csv", index=False)

print("\n=== PYSPARK PHYSICAL & LOGICAL EXECUTION PLAN ===")
sp_out.explain(mode="formatted")

PySpark local[*] runtime: 11.26s
                metric        polars         spark  match
                n_rows  1.682738e+06  1.682738e+06   True
      mean EQ_TO_ASSET  1.067348e-01  1.067348e-01   True
     nulls EQ_TO_ASSET  4.139000e+03  4.139000e+03   True
        mean NPL_RATIO  1.855346e-02  1.855346e-02   True
       nulls NPL_RATIO  2.531000e+03  2.531000e+03   True
 mean D_EQ_TO_ASSET_4Q -2.463945e-03 -2.463945e-03   True
nulls D_EQ_TO_ASSET_4Q  9.878100e+04  9.878100e+04   True

ALL MATCH: True

=== PYSPARK PHYSICAL & LOGICAL EXECUTION PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (14)
+- Project (13)
   +- Window (12)
      +- Sort (11)
         +- Exchange (10)
            +- Project (9)
               +- Filter (8)
                  +- BroadcastHashJoin LeftOuter BuildRight (7)
                     :- Project (2)
                     :  +- Scan parquet  (1)
                     +- BroadcastExchange (6)
                        +- Project (5)
                           

## Parts 4-5 - Labels, features, walk-forward validation, cost-based threshold, Spark MLlib

### Choice of the temporal split (at least 50 failures in the test window)

In [ ]:
import polars as pl
from pathlib import Path

PROC_DIR = Path("/content/drive/MyDrive/bft65b_fdic_project/data_processed")
f = (pl.read_parquet(PROC_DIR / "failures_clean.parquet")
     .filter(pl.col("RESTYPE") == "FAILURE")
     .with_columns((pl.col("FAIL_DT").dt.year() * 4 + (pl.col("FAIL_DT").dt.month() - 1) // 3 + 1).alias("FQ")))

LAST_FQ = 2025 * 4 + 2 + 4   # nhãn của q=2025Q2 nhìn tới 2026Q2
for dev_end_year in (2010, 2011, 2012):
    D = dev_end_year * 4 + 4
    t0 = D + 5                      # chừa khoảng cách: nhãn quý D nhìn tới D+4
    n = (f.filter((pl.col("FQ") >= t0 + 1) & (pl.col("FQ") <= LAST_FQ))
           .select(pl.col("CERT").n_unique()).item())
    print(f"Dev kết thúc {dev_end_year}Q4 -> Test từ {t0 // 4}Q{t0 % 4 or 4}: {n} ngân hàng phá sản")

print("\nPhá sản theo năm từ 2012:")
print(f.filter(pl.col("FAIL_DT").dt.year() >= 2012)
        .group_by(pl.col("FAIL_DT").dt.year().alias("year")).agg(pl.col("CERT").n_unique().alias("n"))
        .sort("year"))

Dev kết thúc 2010Q4 -> Test từ 2012Q1: 117 ngân hàng phá sản
Dev kết thúc 2011Q4 -> Test từ 2013Q1: 78 ngân hàng phá sản
Dev kết thúc 2012Q4 -> Test từ 2014Q1: 53 ngân hàng phá sản

Phá sản theo năm từ 2012:
shape: (12, 2)
┌──────┬─────┐
│ year ┆ n   │
│ ---  ┆ --- │
│ i32  ┆ u32 │
╞══════╪═════╡
│ 2012 ┆ 51  │
│ 2013 ┆ 24  │
│ 2014 ┆ 18  │
│ 2015 ┆ 8   │
│ 2016 ┆ 5   │
│ …    ┆ …   │
│ 2020 ┆ 4   │
│ 2023 ┆ 5   │
│ 2024 ┆ 2   │
│ 2025 ┆ 2   │
│ 2026 ┆ 4   │
└──────┴─────┘


### Labels, features, walk-forward validation (gap of 4 quarters) and cost-based threshold

In [ ]:
import json, time
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

WORK_DIR = Path("/content/drive/MyDrive/bft65b_fdic_project")
PROC_DIR = WORK_DIR / "data_processed"
SEED = 42
np.random.seed(SEED)

# ---------- 1. Dựng nhãn + Dev set (KHÔNG đụng Test) ----------
panel = pl.read_parquet(PROC_DIR / "panel_features_base.parquet")
fails = pl.read_parquet(PROC_DIR / "failures_clean.parquet").filter(pl.col("RESTYPE") == "FAILURE")

def qidx(col):  # năm*4 + quý
    return pl.col(col).dt.year() * 4 + (pl.col(col).dt.month() - 1) // 3 + 1

DEV_MAX_Q = 2012 * 4 + 4
panel = panel.with_columns(qidx("REP_DT").alias("Q_IDX")).filter(pl.col("Q_IDX") <= DEV_MAX_Q)

fails_q = (fails.with_columns(qidx("FAIL_DT").alias("FAIL_Q_IDX"))
           .select(["CERT", "FAIL_Q_IDX", "COST", "FAIL_DT"]))

pos_keys = (panel.select(["CERT", "REP_DT", "Q_IDX"])
            .join(fails_q, on="CERT", how="inner")
            .filter((pl.col("FAIL_Q_IDX") > pl.col("Q_IDX")) & (pl.col("FAIL_Q_IDX") <= pl.col("Q_IDX") + 4))
            .select(["CERT", "REP_DT"]).unique().with_columns(pl.lit(1).alias("TARGET_4Q")))

dev = (panel.join(pos_keys, on=["CERT", "REP_DT"], how="left")
       .with_columns(pl.col("TARGET_4Q").fill_null(0).cast(pl.Int8)))

feature_cols = [
    "ASSET", "LOG_ASSET", "EQ_TO_ASSET", "NPL_RATIO", "RESERVE_TO_LOANS",
    "LIQ_RATIO", "BROKERED_DEP_RATIO", "RE_LOAN_SHARE",
    "D_EQ_TO_ASSET_1Q", "D_EQ_TO_ASSET_4Q", "D_NPL_4Q", "D_ROA_4Q",
    "VOL_ROA_4Q", "ASSET_GROWTH_4Q",
    "ROA", "ROE", "NIMY", "RBC1AAJ", "RBCRWAJ", "P9ASSET", "NAASSET", "NTLNLS", "ORE",
]
dev_pd = dev.select(["CERT", "Q_IDX", "TARGET_4Q"] + feature_cols).to_pandas()
print(f"Dev: {len(dev_pd):,} rows | positives: {int(dev_pd.TARGET_4Q.sum()):,}")

# ---------- 2. Chi phí bỏ sót (FN) ước lượng từ dữ liệu, chỉ dùng failures trong Dev ----------
dev_fail_cost = (fails_q.filter(pl.col("FAIL_Q_IDX") <= DEV_MAX_Q)
                 .group_by("CERT").agg(pl.col("COST").sum()).to_pandas())
dev_fail_cost = dev_fail_cost[dev_fail_cost["COST"] > 0]
COST_FN = float(dev_fail_cost["COST"].mean())  # nghìn USD / ngân hàng phá sản
print(f"COST_FN (mean DIF cost per failed bank, Dev): {COST_FN:,.0f}k USD "
      f"(n={len(dev_fail_cost)}; median={dev_fail_cost['COST'].median():,.0f}k)")

# ---------- 3. Tiền xử lý: mọi thống kê chỉ fit trên train của từng fold ----------
class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.005, upper=0.995):
        self.lower, self.upper = lower, upper
    def fit(self, X, y=None):
        A = np.asarray(X, dtype=float).copy()
        A[~np.isfinite(A)] = np.nan
        self.lo_ = np.nanquantile(A, self.lower, axis=0)
        self.hi_ = np.nanquantile(A, self.upper, axis=0)
        return self
    def transform(self, X):
        A = np.asarray(X, dtype=float).copy()
        A[~np.isfinite(A)] = np.nan
        return np.clip(A, self.lo_, self.hi_)

def make_pipe(model):
    return Pipeline([("win", Winsorizer()),
                     ("imp", SimpleImputer(strategy="median")),
                     ("sc", RobustScaler()),
                     ("clf", model)])

def make_models():
    return {
        "LogReg": make_pipe(LogisticRegression(max_iter=2000, random_state=SEED)),
        "HGB": make_pipe(HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05,
                                                        max_depth=6, random_state=SEED)),
    }

# ---------- 4. Walk-forward (expanding window) với GAP 4 quý ----------
folds = [(1988, 1989), (1990, 1991), (1992, 1993), (1994, 1995),(1996, 2007), (2008, 2009), (2010, 2012)]  # khoảng năm của val
oof, fold_rows = [], []
t0 = time.perf_counter()
for (y0, y1) in folds:
    v0, v1 = y0 * 4 + 1, y1 * 4 + 4
    tr = dev_pd[dev_pd.Q_IDX <= v0 - 5]            # nhãn của q nhìn tới q+4 < v0  => không chồng val
    va = dev_pd[(dev_pd.Q_IDX >= v0) & (dev_pd.Q_IDX <= v1)]
    if tr.TARGET_4Q.sum() == 0 or va.TARGET_4Q.sum() == 0:
        print(f"Bỏ qua fold {y0}-{y1}: thiếu positive"); continue
    part = va[["CERT", "Q_IDX", "TARGET_4Q"]].copy()
    for name, pipe in make_models().items():
        pipe.fit(tr[feature_cols], tr.TARGET_4Q.values)
        part[name] = pipe.predict_proba(va[feature_cols])[:, 1]
        fold_rows.append({"fold": f"{y0}-{y1}", "model": name,
                          "n_train": len(tr), "pos_train": int(tr.TARGET_4Q.sum()),
                          "n_val": len(va), "pos_val": int(va.TARGET_4Q.sum()),
                          "ROC_AUC": roc_auc_score(va.TARGET_4Q, part[name]),
                          "PR_AUC": average_precision_score(va.TARGET_4Q, part[name])})
    oof.append(part)
    print(f"fold {y0}-{y1} xong ({time.perf_counter() - t0:.0f}s)")

oof = pd.concat(oof, ignore_index=True)
print("\n=== KẾT QUẢ THEO FOLD (walk-forward, gap 4 quý) ===")
print(pd.DataFrame(fold_rows).round(4).to_string(index=False))

# ---------- 5. Chọn ngưỡng theo chi phí trên OOF gộp, riêng từng mô hình ----------
grid = np.unique(np.concatenate([np.logspace(-4, -0.01, 120), [0.5]]))
y = oof.TARGET_4Q.values
RATIOS = [1/20, 1/50, 1/100, 1/200, 1/500]   # COST_FP / COST_FN (giả định, có bảng nhạy cảm)
BASE_RATIO = 1/100

def best_threshold(p, ratio):
    costs = []
    for th in grid:
        pred = p >= th
        fn = int(((~pred) & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
        costs.append(fn * COST_FN + fp * COST_FN * ratio)
    i = int(np.argmin(costs))
    return float(grid[i]), float(costs[i])

sens = []
for name in ["LogReg", "HGB"]:
    p = oof[name].values
    for r in RATIOS:
        th, c = best_threshold(p, r)
        pred = p >= th
        sens.append({"model": name, "FP/FN cost ratio": f"1:{int(round(1/r))}", "threshold": round(th, 4),
                     "recall": round(((pred) & (y == 1)).sum() / max(1, (y == 1).sum()), 3),
                     "alert_rate_%": round(100 * pred.mean(), 2),
                     "FP": int((pred & (y == 0)).sum()), "FN": int(((~pred) & (y == 1)).sum())})
print("\n=== BẢNG NHẠY CẢM NGƯỠNG THEO TỶ LỆ CHI PHÍ (OOF, chỉ Dev) ===")
print(pd.DataFrame(sens).to_string(index=False))

thresholds = {n: best_threshold(oof[n].values, BASE_RATIO)[0] for n in ["LogReg", "HGB"]}
print("\nNgưỡng chốt (tỷ lệ cơ sở 1:100):", thresholds)
json.dump({"thresholds": thresholds, "COST_FN": COST_FN, "BASE_RATIO": BASE_RATIO,
           "folds": folds, "feature_cols": feature_cols},
          open(PROC_DIR / "model_config_frozen.json", "w"), indent=1)
pd.DataFrame(fold_rows).to_csv(WORK_DIR / "benchmarks" / "walkforward_folds.csv", index=False)
pd.DataFrame(sens).to_csv(WORK_DIR / "benchmarks" / "threshold_sensitivity.csv", index=False)

Dev: 1,387,614 rows | positives: 10,649
COST_FN (mean DIF cost per failed bank, Dev): 72,866k USD (n=2694; median=12,399k)
fold 1988-1989 xong (22s)
fold 1990-1991 xong (58s)
fold 1992-1993 xong (106s)
fold 1994-1995 xong (179s)
fold 1996-2007 xong (263s)
fold 2008-2009 xong (362s)
fold 2010-2012 xong (547s)

=== KẾT QUẢ THEO FOLD (walk-forward, gap 4 quý) ===
     fold  model  n_train  pos_train  n_val  pos_val  ROC_AUC  PR_AUC
1988-1989 LogReg   216033       1799 133104     3452   0.9290  0.3971
1988-1989    HGB   216033       1799 133104     3452   0.9533  0.4069
1990-1991 LogReg   354142       4506 122710     1935   0.9518  0.2498
1990-1991    HGB   354142       4506 122710     1935   0.9697  0.3078
1992-1993 LogReg   482300       7243 111512      465   0.9715  0.2032
1992-1993    HGB   482300       7243 111512      465   0.9792  0.3809
1994-1995 LogReg   599266       8400 100991       68   0.9815  0.1882
1994-1995    HGB   599266       8400 100991       68   0.9865  0.2835
1996-20

### Part 5 - Spark MLlib demo (development data only)

In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression as SparkLR
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = (SparkSession.builder.master("local[*]").appName("BFT65B_MLlib_Demo")
         .config("spark.driver.memory", "4g").getOrCreate())
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# Fold 1990-1991 của walk-forward, có gap 4 quý, chỉ dùng dữ liệu Dev
v0 = 1990 * 4 + 1
tr = dev_pd[dev_pd.Q_IDX <= v0 - 5]
va = dev_pd[(dev_pd.Q_IDX >= v0) & (dev_pd.Q_IDX <= 1991 * 4 + 4)]

# Bỏ cột hoàn toàn rỗng trên train (Spark Imputer sẽ báo lỗi nếu cột toàn NaN)
clean = lambda d: d[feature_cols].replace([np.inf, -np.inf], np.nan)
usable = [c for c in feature_cols if clean(tr)[c].notna().any()]
print("Cột bị loại (rỗng trên train):", sorted(set(feature_cols) - set(usable)))

# Winsorize: phân vị chỉ tính trên train
win = Winsorizer().fit(tr[feature_cols])
def to_sdf(d):
    X = pd.DataFrame(win.transform(d[feature_cols]), columns=feature_cols)[usable]
    X["label"] = d.TARGET_4Q.astype(float).values
    return spark.createDataFrame(X)
sp_tr, sp_va = to_sdf(tr), to_sdf(va)

imp_cols = [c + "_imp" for c in usable]
pipe = SparkPipeline(stages=[
    Imputer(inputCols=usable, outputCols=imp_cols, strategy="median"),
    VectorAssembler(inputCols=imp_cols, outputCol="raw"),
    StandardScaler(inputCol="raw", outputCol="features", withMean=True, withStd=True),
    SparkLR(featuresCol="features", labelCol="label", maxIter=100, regParam=1e-5),
])
model = pipe.fit(sp_tr)          # mọi thống kê (median, mean, std) chỉ học từ train
pred = model.transform(sp_va)

roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",
                                    metricName="areaUnderROC").evaluate(pred)
pr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",
                                   metricName="areaUnderPR").evaluate(pred)
print(f"Spark MLlib LogReg (fold 1990-1991) -> ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f}")
print("sklearn LogReg (cùng fold)         -> ROC-AUC: 0.9518 | PR-AUC: 0.2498")

Cột bị loại (rỗng trên train): []
Spark MLlib LogReg (fold 1990-1991) -> ROC-AUC: 0.9521 | PR-AUC: 0.2477
sklearn LogReg (cùng fold)         -> ROC-AUC: 0.9518 | PR-AUC: 0.2498


### Part 4 - Feature/label table and automated leakage test

In [ ]:
import time, json, datetime
import numpy as np, pandas as pd, polars as pl
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

W = Path("/content/drive/MyDrive/bft65b_fdic_project")
P, R = W / "data_processed", W / "results"
R.mkdir(exist_ok=True)

cfg = json.load(open(P / "model_config_frozen.json"))
FEATURES = cfg["feature_cols"]
FORBIDDEN = {"ACTIVE", "CHANGEC1", "END_DT", "EST_DT", "IS_MERGER_EXIT", "PANEL_STATUS", "BKCLASS", "TARGET_4Q"}
assert not (set(FEATURES) & FORBIDDEN), "Feature list contains a post-event / label-like column"

def qidx(c):
    return pl.col(c).dt.year() * 4 + (pl.col(c).dt.month() - 1) // 3 + 1

DEV_MAX_Q, TEST_MIN_Q, LAST_Q = 2012 * 4 + 4, 2014 * 4 + 1, 2025 * 4 + 2

# ---------- 1. Bảng đặc trưng + nhãn ----------
panel = (pl.read_parquet(P / "panel_features_base.parquet")
         .with_columns(qidx("REP_DT").alias("Q_IDX"))
         .filter(pl.col("Q_IDX") <= LAST_Q))
fails = (pl.read_parquet(P / "failures_clean.parquet").filter(pl.col("RESTYPE") == "FAILURE")
         .with_columns(qidx("FAIL_DT").alias("FAIL_Q_IDX")).select(["CERT", "FAIL_Q_IDX"]))

pos = (panel.select(["CERT", "REP_DT", "Q_IDX"]).join(fails, on="CERT", how="inner")
       .filter((pl.col("FAIL_Q_IDX") > pl.col("Q_IDX")) & (pl.col("FAIL_Q_IDX") <= pl.col("Q_IDX") + 4))
       .group_by(["CERT", "REP_DT"])
       .agg((pl.col("FAIL_Q_IDX").min() - pl.col("Q_IDX").first()).alias("Q_TO_FAIL")))

table = (panel.join(pos, on=["CERT", "REP_DT"], how="left")
         .with_columns([
             pl.col("Q_TO_FAIL").is_not_null().cast(pl.Int8).alias("TARGET_4Q"),
             pl.when(pl.col("Q_IDX") <= DEV_MAX_Q).then(pl.lit("dev"))
               .when(pl.col("Q_IDX") >= TEST_MIN_Q).then(pl.lit("test"))
               .otherwise(pl.lit("gap")).alias("SPLIT")])
         .select(["CERT", "REP_DT", "Q_IDX", "SPLIT", "TARGET_4Q"] + FEATURES))   # Q_TO_FAIL không lưu (là thông tin tương lai)
table.write_parquet(P / "features_labels.parquet", compression="zstd")

summ = (table.group_by("SPLIT").agg(pl.len().alias("rows"), pl.col("TARGET_4Q").sum().alias("positive_rows"),
                                    pl.col("CERT").n_unique().alias("banks")).sort("SPLIT"))
print("=== BẢNG ĐẶC TRƯNG + NHÃN ===")
print(summ)
got = {r["SPLIT"]: (r["rows"], r["positive_rows"]) for r in summ.iter_rows(named=True)}
expected = {"dev": (1_387_614, 10_649), "test": (249_606, 175)}
for k, v in expected.items():
    print(f"{k}: kỳ vọng {v}, thực tế {got[k]} ->", "KHỚP" if got[k] == v else "KHÔNG KHỚP!")

dist = pos.group_by("Q_TO_FAIL").agg(pl.len().alias("positive_rows")).sort("Q_TO_FAIL")
print("\nSố quý từ điểm quan sát tới ngày phá sản (chỉ các dòng positive):")
print(dist)
dist.write_csv(R / "label_horizon_distribution.csv")

# ---------- 2. Kiểm tra rò rỉ bằng cách cắt dữ liệu ----------
fin = pl.scan_parquet(W / "benchmarks/financials_unified_zstd.parquet")
inst = pl.scan_parquet(P / "institutions_clean.parquet").select(["CERT", "END_DT"])

def compute(fin_lf, leaky=False):
    lf = (fin_lf
          .with_columns([pl.col("REPDTE").str.strptime(pl.Date, "%Y%m%d").alias("REP_DT"),
                         pl.when(pl.col("LNLSGR") < 0).then(None).otherwise(pl.col("LNLSGR")).alias("LNLSGR")])
          .join(inst, on="CERT", how="left")
          .filter(pl.col("END_DT").is_null() | (pl.col("REP_DT") <= pl.col("END_DT")))
          .sort(["CERT", "REP_DT"])
          .with_columns([
              (pl.col("EQ") / pl.col("ASSET")).alias("EQ_TO_ASSET"),
              (pl.col("NCLNLS") / pl.col("LNLSGR").clip(lower_bound=1.0)).alias("NPL_RATIO"),
              (pl.col("LNATRES") / pl.col("LNLSGR").clip(lower_bound=1.0)).alias("RESERVE_TO_LOANS"),
              ((pl.col("CHBAL") + pl.col("SC")) / pl.col("ASSET")).alias("LIQ_RATIO"),
              (pl.col("BRO") / pl.col("DEP").clip(lower_bound=1.0)).alias("BROKERED_DEP_RATIO"),
              (pl.col("LNRE") / pl.col("LNLSGR").clip(lower_bound=1.0)).alias("RE_LOAN_SHARE"),
              pl.col("ASSET").log().alias("LOG_ASSET")])
          .with_columns([
              (pl.col("EQ_TO_ASSET") - pl.col("EQ_TO_ASSET").shift(1).over("CERT")).alias("D_EQ_TO_ASSET_1Q"),
              (pl.col("EQ_TO_ASSET") - pl.col("EQ_TO_ASSET").shift(4).over("CERT")).alias("D_EQ_TO_ASSET_4Q"),
              (pl.col("NPL_RATIO") - pl.col("NPL_RATIO").shift(4).over("CERT")).alias("D_NPL_4Q"),
              (pl.col("ROA") - pl.col("ROA").shift(4).over("CERT")).alias("D_ROA_4Q"),
              pl.col("ROA").rolling_std(window_size=4, min_samples=2).over("CERT").alias("VOL_ROA_4Q"),
              pl.col("ASSET").pct_change(n=4).over("CERT").alias("ASSET_GROWTH_4Q")]))
    if leaky:   # ĐỐI CHỨNG ÂM: đặc trưng cố tình dùng tương lai
        lf = lf.with_columns(pl.col("ASSET").shift(-1).over("CERT").alias("LEAKY_NEXT_ASSET"))
    return lf.select(["CERT", "REP_DT"] + FEATURES + (["LEAKY_NEXT_ASSET"] if leaky else [])).collect()

def compare(A, B, cols):
    j = A.join(B, on=["CERT", "REP_DT"], how="inner", suffix="_b")
    out = {}
    for c in cols:
        x, y = j[c].to_numpy().astype(float), j[c + "_b"].to_numpy().astype(float)
        out[c] = bool(np.allclose(x, y, rtol=1e-9, atol=1e-12, equal_nan=True))
    return j.height, out

t0 = time.perf_counter()
A = compute(fin, leaky=True)
print(f"\nĐã tính đặc trưng trên dữ liệu đầy đủ: {A.height:,} dòng ({time.perf_counter() - t0:.0f}s)")

# (a) tính lại có trùng với bảng đã lưu dùng cho mô hình không
saved = pl.read_parquet(P / "panel_features_base.parquet").select(["CERT", "REP_DT"] + FEATURES)
n_saved, ok_saved = compare(A.select(["CERT", "REP_DT"] + FEATURES), saved, FEATURES)
print("Tính lại so với bảng đã lưu:", n_saved, "dòng, tất cả khớp:", all(ok_saved.values()))

rows = []
for T in ["1995-12-31", "2008-12-31", "2013-12-31"]:
    d = datetime.date.fromisoformat(T)
    B = compute(fin.filter(pl.col("REPDTE") <= T.replace("-", "")), leaky=True)
    A_cut = A.filter(pl.col("REP_DT") <= d)
    n, ok = compare(A_cut, B, FEATURES + ["LEAKY_NEXT_ASSET"])
    rows.append({"truncate_at": T, "rows_compared": n, "rows_equal_in_both": (A_cut.height == B.height),
                 "all_features_identical": all(ok[c] for c in FEATURES),
                 "features_that_differ": ",".join([c for c in FEATURES if not ok[c]]) or "none",
                 "negative_control_detected": (not ok["LEAKY_NEXT_ASSET"])})
lk = pd.DataFrame(rows)
print("\n=== KIỂM TRA RÒ RỈ BẰNG CÁCH CẮT DỮ LIỆU ===")
print(lk.to_string(index=False))
lk.to_csv(R / "leakage_truncation_test.csv", index=False)

# ---------- 3. Từ điển đặc trưng ----------
doc = {
 "ASSET": ("Total assets, thousand USD (Call Report)", "Size"),
 "LOG_ASSET": ("ln(ASSET)", "Size, reduces skew"),
 "EQ_TO_ASSET": ("EQ / ASSET", "Capital adequacy (leverage)"),
 "NPL_RATIO": ("NCLNLS / max(LNLSGR, 1)", "Asset quality (noncurrent loans)"),
 "RESERVE_TO_LOANS": ("LNATRES / max(LNLSGR, 1)", "Loss-absorbing reserves"),
 "LIQ_RATIO": ("(CHBAL + SC) / ASSET", "Liquidity"),
 "BROKERED_DEP_RATIO": ("BRO / max(DEP, 1)", "Funding fragility"),
 "RE_LOAN_SHARE": ("LNRE / max(LNLSGR, 1)", "Concentration in real-estate lending"),
 "D_EQ_TO_ASSET_1Q": ("EQ_TO_ASSET minus its value 1 quarter earlier", "Capital trend"),
 "D_EQ_TO_ASSET_4Q": ("EQ_TO_ASSET minus its value 4 quarters earlier", "Capital trend"),
 "D_NPL_4Q": ("NPL_RATIO minus its value 4 quarters earlier", "Deterioration of asset quality"),
 "D_ROA_4Q": ("ROA minus its value 4 quarters earlier", "Earnings trend"),
 "VOL_ROA_4Q": ("Rolling std of ROA over 4 quarters (min 2)", "Earnings volatility"),
 "ASSET_GROWTH_4Q": ("ASSET / ASSET 4 quarters earlier - 1", "Rapid-growth risk"),
 "ROA": ("Return on assets, % (as reported)", "Earnings"),
 "ROE": ("Return on equity, % (as reported)", "Earnings"),
 "NIMY": ("Net interest margin, % (as reported)", "Earnings"),
 "RBC1AAJ": ("Regulatory capital ratio (as reported)", "Capital"),
 "RBCRWAJ": ("Regulatory risk-weighted capital ratio (as reported)", "Capital"),
 "P9ASSET": ("Call Report asset-quality field (check FDIC dictionary)", "Asset quality"),
 "NAASSET": ("Call Report asset-quality field (check FDIC dictionary)", "Asset quality"),
 "NTLNLS": ("Call Report asset-quality field (check FDIC dictionary)", "Asset quality"),
 "ORE": ("Other real estate owned (check FDIC dictionary)", "Asset quality"),
}
fd = pd.DataFrame([{"feature": f, "definition": doc[f][0], "CAMELS dimension / rationale": doc[f][1]} for f in FEATURES])
fd.to_csv(R / "feature_dictionary.csv", index=False)
print("\nĐã lưu: features_labels.parquet, leakage_truncation_test.csv, label_horizon_distribution.csv, feature_dictionary.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== BẢNG ĐẶC TRƯNG + NHÃN ===
shape: (3, 4)
┌───────┬─────────┬───────────────┬───────┐
│ SPLIT ┆ rows    ┆ positive_rows ┆ banks │
│ ---   ┆ ---     ┆ ---           ┆ ---   │
│ str   ┆ u32     ┆ i64           ┆ u32   │
╞═══════╪═════════╪═══════════════╪═══════╡
│ dev   ┆ 1387614 ┆ 10649         ┆ 23962 │
│ gap   ┆ 27989   ┆ 79            ┆ 7105  │
│ test  ┆ 249606  ┆ 175           ┆ 6901  │
└───────┴─────────┴───────────────┴───────┘
dev: kỳ vọng (1387614, 10649), thực tế (1387614, 10649) -> KHỚP
test: kỳ vọng (249606, 175), thực tế (249606, 175) -> KHỚP

Số quý từ điểm quan sát tới ngày phá sản (chỉ các dòng positive):
shape: (4, 2)
┌───────────┬───────────────┐
│ Q_TO_FAIL ┆ positive_rows │
│ ---       ┆ ---           │
│ i32       ┆ u32           │
╞═══════════╪═══════════════╡
│ 1         ┆ 2529          │
│ 2         ┆ 2811          │
│ 3         ┆ 279

## Final test evaluation (single frozen configuration)

In [ ]:
import json, time
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score

try:
    spark.stop()   # giải phóng RAM cho HGB nếu Spark còn mở
except NameError:
    pass

WORK_DIR = Path("/content/drive/MyDrive/bft65b_fdic_project")
PROC_DIR = WORK_DIR / "data_processed"
LOCK = WORK_DIR / "TEST_USED.lock"
if LOCK.exists():
    raise RuntimeError("Tập test đã được dùng! Không chạy lại. Xem test_results_FINAL.json")

cfg = json.load(open(PROC_DIR / "model_config_frozen.json"))
thr, COST_FN = cfg["thresholds"], cfg["COST_FN"]
feature_cols = cfg["feature_cols"]

def qidx(col):
    return pl.col(col).dt.year() * 4 + (pl.col(col).dt.month() - 1) // 3 + 1

DEV_MAX_Q = 2012 * 4 + 4
TEST_MIN_Q, TEST_MAX_Q = 2014 * 4 + 1, 2025 * 4 + 2
assert len(dev_pd) == 1_387_614 and dev_pd.Q_IDX.max() == DEV_MAX_Q, "dev_pd không khớp cấu hình đã đóng băng"
assert TEST_MIN_Q - DEV_MAX_Q >= 5, "Gap giữa Dev và Test phải >= 4 quý"

# ---------- 1. Dựng nhãn Test ----------
panel = pl.read_parquet(PROC_DIR / "panel_features_base.parquet").with_columns(qidx("REP_DT").alias("Q_IDX"))
fails = (pl.read_parquet(PROC_DIR / "failures_clean.parquet").filter(pl.col("RESTYPE") == "FAILURE")
         .with_columns(qidx("FAIL_DT").alias("FAIL_Q_IDX")).select(["CERT", "FAIL_Q_IDX"]))

in_test = panel.filter((pl.col("Q_IDX") >= TEST_MIN_Q) & (pl.col("Q_IDX") <= TEST_MAX_Q))
pos_keys = (in_test.select(["CERT", "REP_DT", "Q_IDX"]).join(fails, on="CERT", how="inner")
            .filter((pl.col("FAIL_Q_IDX") > pl.col("Q_IDX")) & (pl.col("FAIL_Q_IDX") <= pl.col("Q_IDX") + 4))
            .select(["CERT", "REP_DT"]).unique().with_columns(pl.lit(1).alias("TARGET_4Q")))
test = (in_test.join(pos_keys, on=["CERT", "REP_DT"], how="left")
        .with_columns(pl.col("TARGET_4Q").fill_null(0).cast(pl.Int8))
        .select(["CERT", "Q_IDX", "TARGET_4Q"] + feature_cols).to_pandas())
y_test = test.TARGET_4Q.values
n_failed = int(test.loc[test.TARGET_4Q == 1, "CERT"].nunique())

# ---------- 2. PRE-CHECK (chưa chạm mô hình, không lộ hiệu suất) ----------
raw = pd.read_parquet(WORK_DIR / "data_raw" / "fdic" / "failures.parquet")
raw["FAILYR_NUM"] = pd.to_numeric(raw["FAILYR"], errors="coerce")
miss = raw[raw.CERT.isna() & (raw.FAILYR_NUM >= 2014) & (raw.RESTYPE == "FAILURE")]
print(f"Test: {len(test):,} rows | positive rows: {int(y_test.sum()):,} | failed banks in Test: {n_failed}")
print(f"Dev : {len(dev_pd):,} rows | positive rows: {int(dev_pd.TARGET_4Q.sum()):,}")
print(f"Vụ FAILURE từ 2014 bị mất do thiếu CERT trong bảng failures: {len(miss)}")
assert n_failed >= 50, f"Test chỉ có {n_failed} ngân hàng phá sản (<50)."

# ---------- 3. Fit trên TOÀN BỘ Dev, chấm Test MỘT LẦN ----------
X_dev, y_dev = dev_pd[feature_cols], dev_pd.TARGET_4Q.values
results, probs = [], {}
for name, pipe in make_models().items():
    pipe.fit(X_dev, y_dev)
    p = pipe.predict_proba(test[feature_cols])[:, 1]
    probs[name] = p
    pred = p >= thr[name]
    tp = int((pred & (y_test == 1)).sum()); fp = int((pred & (y_test == 0)).sum())
    fn = int((~pred & (y_test == 1)).sum()); tn = int((~pred & (y_test == 0)).sum())
    row = {"model": name, "threshold": round(thr[name], 5),
           "ROC_AUC": roc_auc_score(y_test, p), "PR_AUC": average_precision_score(y_test, p),
           "recall": tp / max(1, tp + fn), "precision": tp / max(1, tp + fp),
           "alert_rate_%": 100 * pred.mean(), "TP": tp, "FP": fp, "FN": fn, "TN": tn}
    for r in [1/20, 1/100, 1/500]:
        row[f"cost_1:{int(round(1/r))}_mUSD"] = (fn * COST_FN + fp * COST_FN * r) / 1000
    results.append(row)

n_pos, n_neg = int(y_test.sum()), int((y_test == 0).sum())
refs = {"no_alert": {f"cost_1:{k}_mUSD": n_pos * COST_FN / 1000 for k in (20, 100, 500)},
        "alert_all": {f"cost_1:{k}_mUSD": n_neg * COST_FN / k / 1000 for k in (20, 100, 500)}}

# ---------- 4. LƯU + KHÓA NGAY (trước bootstrap) ----------
out = {"dev_rows": len(dev_pd), "test_rows": len(test), "positive_rows": n_pos,
       "distinct_failed_banks": n_failed, "missing_cert_failures_2014plus": int(len(miss)),
       "results": results, "references": refs}
json.dump(out, open(WORK_DIR / "test_results_FINAL.json", "w"), indent=1, default=float)
pd.DataFrame(results).to_csv(WORK_DIR / "benchmarks" / "test_results_FINAL.csv", index=False)
LOCK.write_text(f"Test evaluated once at {time.ctime()}")

print("\n=== KẾT QUẢ TEST 2014Q1-2025Q2 (đánh giá đúng 1 lần) ===")
print(pd.DataFrame(results).T.to_string(header=False))
print("\nTham chiếu chi phí (triệu USD):", {k: {a: round(b) for a, b in v.items()} for k, v in refs.items()})
print(f"PR-AUC của mô hình ngẫu nhiên = tỷ lệ positive = {y_test.mean():.5f}")
print("Đã lưu và khóa tập test.")

# ---------- 5. Bootstrap theo ngân hàng (chỉ đo độ bất định, không chỉnh gì) ----------
groups = test.groupby("CERT").indices
certs = np.array(list(groups.keys()))
rng = np.random.default_rng(42)
ci = {n: {"roc": [], "pr": []} for n in probs}
for _ in range(100):
    idx = np.concatenate([groups[c] for c in rng.choice(certs, size=len(certs), replace=True)])
    if y_test[idx].sum() == 0:
        continue
    for n, p in probs.items():
        ci[n]["roc"].append(roc_auc_score(y_test[idx], p[idx]))
        ci[n]["pr"].append(average_precision_score(y_test[idx], p[idx]))
ci_out = {n: {"ROC_CI95": list(np.percentile(v["roc"], [2.5, 97.5])),
              "PR_CI95": list(np.percentile(v["pr"], [2.5, 97.5]))} for n, v in ci.items()}
json.dump(ci_out, open(WORK_DIR / "test_bootstrap_ci.json", "w"), indent=1)
print("\nBootstrap 95% CI (100 lần, theo ngân hàng):", json.dumps(ci_out, indent=1))

Test: 249,606 rows | positive rows: 175 | failed banks in Test: 53
Dev : 1,387,614 rows | positive rows: 10,649
Vụ FAILURE từ 2014 bị mất do thiếu CERT trong bảng failures: 0

=== KẾT QUẢ TEST 2014Q1-2025Q2 (đánh giá đúng 1 lần) ===
model                  LogReg           HGB
threshold             0.00475       0.00237
ROC_AUC               0.95231       0.93374
PR_AUC               0.352819      0.231751
recall               0.788571           0.8
precision            0.060766      0.031482
alert_rate_%         0.909834      1.781608
TP                        138           140
FP                       2133          4307
FN                         37            35
TN                     247298        245124
cost_1:20_mUSD   10467.246886  18242.083243
cost_1:100_mUSD   4250.292453   5688.673612
cost_1:500_mUSD   3006.901566   3177.991686

Tham chiếu chi phí (triệu USD): {'no_alert': {'cost_1:20_mUSD': 12752, 'cost_1:100_mUSD': 12752, 'cost_1:500_mUSD': 12752}, 'alert_all': {'cost_1:20_m

> Note on the test set: the evaluation cell above was executed more than once, always with the same frozen configuration (`model_config_frozen.json`) and the same random seeds. All outputs were identical and no modelling decision was changed between runs. Before the current temporal split was fixed, an earlier exploratory run touched the 2008-2012 period; that period was therefore moved into the development set and the test window moved to 2014Q1-2025Q2.

## Utilities (delete before submission)

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

W = Path("/content/drive/MyDrive/bft65b_fdic_project")
R = W / "results"; R.mkdir(exist_ok=True)
for p in ["test_results_FINAL.json", "test_bootstrap_ci.json", "data_processed/model_config_frozen.json",
          "benchmarks/walkforward_folds.csv", "benchmarks/threshold_sensitivity.csv",
          "benchmarks/polars_vs_spark_check.csv", "benchmarks/polars_vs_pandas_scaling.csv",
          "benchmarks/polars_vs_pandas_scaling_v2.csv", "benchmarks/test_results_FINAL.csv",
          "benchmarks/partition_pruning.csv"]:
    f = W / p
    if f.exists():
        shutil.copy(f, R / f.name); print("đã chép  ", p)
    else:
        print("KHÔNG CÓ", p)

shutil.make_archive("/content/results_backup", "zip", R)
files.download("/content/results_backup.zip")
print("\nCác file trong results/:", sorted(x.name for x in R.iterdir()))

đã chép   test_results_FINAL.json
đã chép   test_bootstrap_ci.json
đã chép   data_processed/model_config_frozen.json
đã chép   benchmarks/walkforward_folds.csv
đã chép   benchmarks/threshold_sensitivity.csv
đã chép   benchmarks/polars_vs_spark_check.csv
đã chép   benchmarks/polars_vs_pandas_scaling.csv
đã chép   benchmarks/polars_vs_pandas_scaling_v2.csv
đã chép   benchmarks/test_results_FINAL.csv
đã chép   benchmarks/partition_pruning.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Các file trong results/: ['data_quality_report.md', 'dq_anomalies.csv', 'dq_exit_codes_vs_failures.csv', 'dq_key_reconciliation.csv', 'dq_missing_by_decade.csv', 'dq_quarter_gaps.csv', 'feature_dictionary.csv', 'label_horizon_distribution.csv', 'leakage_truncation_test.csv', 'model_config_frozen.json', 'partition_pruning.csv', 'polars_vs_pandas_scaling.csv', 'polars_vs_pandas_scaling_v2.csv', 'polars_vs_spark_check.csv', 'test_bootstrap_ci.json', 'test_results_FINAL.csv', 'test_results_FINAL.json', 'threshold_sensitivity.csv', 'walkforward_folds.csv']


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os, datetime
from pathlib import Path
W = Path("/content/drive/MyDrive/bft65b_fdic_project")
print("Thư mục dự án tồn tại:", W.exists())

t = lambda p: datetime.datetime.fromtimestamp(os.path.getmtime(p)).strftime("%Y-%m-%d %H:%M")

fin = sorted(W.rglob("REPDTE=*.parquet"))
print("Số file quý tìm thấy:", len(fin))
if fin:
    print("Nằm ở:", fin[0].parent)
    print("Financials: từ", min(map(t, fin)), "đến", max(map(t, fin)))

for name in ["manifest_financials.json", "failures.parquet", "institutions.parquet"]:
    for p in W.rglob(name):
        print(name, "->", t(p), "|", p)

Mounted at /content/drive
Thư mục dự án tồn tại: True
Số file quý tìm thấy: 170
Nằm ở: /content/drive/MyDrive/bft65b_fdic_project/data_raw/fdic/financials
Financials: từ 2026-11-19 03:06 đến 2026-11-19 03:22
manifest_financials.json -> 2026-11-19 03:22 | /content/drive/MyDrive/bft65b_fdic_project/data_raw/fdic/manifest_financials.json
failures.parquet -> 2026-11-20 03:39 | /content/drive/MyDrive/bft65b_fdic_project/data_raw/fdic/failures.parquet
institutions.parquet -> 2026-11-20 03:03 | /content/drive/MyDrive/bft65b_fdic_project/data_raw/fdic/institutions.parquet


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import polars as pl
t = pl.read_parquet("/content/drive/MyDrive/bft65b_fdic_project/data_processed/features_labels.parquet")
n_dev = t.filter((pl.col("SPLIT") == "dev") & (pl.col("TARGET_4Q") == 1))["CERT"].n_unique()
print("Failed banks in Dev:", n_dev)

Mounted at /content/drive
Failed banks in Dev: 2779
